# The general 2 mode case of the coherent coupler

See:
- "The Nonlinear Coherent Coupler" by Jensen, https://ieeexplore.ieee.org/document/1131291


In [2]:
from sympy import *

# -- Symbols --

(
    t, t0, j, k, l, m, n, g2, g3, x, y, z, z0, z1, N, M, C
) = symbols('''t, t0, j, k, l, m, n, g2, g3, x, y, z, z0, z1, N, M, C''')

# -- Functions --

pw = Function('pw') # Weierstrass P function
pwp = Function('pwp') # Derivative of Weierstrass P function
zw = Function('zw') # Weierstrass Zeta function
sigma = Function('sigma') # Weierstrass Sigma function
zeta = Function('zeta') # Weierstrass Zeta function
wp = Function('wp') # Weierstrass P function
wpp = Function('\wp\'') # Derivative of Weierstrass P function
Delta = Function('Delta') # Descriminant

q = Function('q')
s = Function('s')
f = Function('f')
u = Function('u')
v = Function('v')
w = Function('w')

Q = Function('Q')
S = Function('S')
R = Function('R')
U = Function('U')
H = Function('H')
T = Function('T')
SomeInteger = Function('SomeInteger')
Det = Function('Det') # Unevaluated determinant

rho = Function('rho')
rhop = Function(r"\rho'", latex_name=r'\rho\'')
phi = Function('phi')
ubar = Function('ubar', latex_name=r'\bar{u}')
vbar = Function('vbar', latex_name=r'\bar{v}')

# -- Indexed Symbols --

a = IndexedBase('a')
b = IndexedBase('b')
c = IndexedBase('c')
d = IndexedBase('d')
p = IndexedBase('p')
r = IndexedBase('r')

K = IndexedBase('K')

alpha = IndexedBase('alpha')
beta = IndexedBase('beta')
mu = IndexedBase('mu')
nu = IndexedBase('nu')
gamma = IndexedBase('gamma')
xi = IndexedBase('xi')
eta = IndexedBase('eta')
epsilon = IndexedBase('epsilon')
iota = IndexedBase('iota')
lamda = IndexedBase('lamda') # special sympy spelling
omega = IndexedBase('omega')
mubar = IndexedBase('mubar', latex_name=r'\bar{\mu}')

Lamda = IndexedBase('Lamda') # special sympy spelling
Omega = IndexedBase('Omega')


wild = Wild('*')


from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# The package containing mpmath expressions for Weierstrass elliptic functions
from weierstrass_modified import Weierstrass
we = Weierstrass()
from mpmath import exp as mpexp
from mpmath import log as mplog

# kth order derivatives of Weierstrass P
from wpk import wpk


# Numeric solutions to diff eqs
from numpy import (
    linspace, absolute, angle, square, real, imag, conj, array as arraynp, 
    concatenate, real_if_close
)
from numpy import vectorize as np_vectorize # not to get confused with vectorise in other packages
from numpy import log as np_log
import scipy.integrate
import matplotlib.pyplot as plt
from random import random, uniform as random_uniform

import numpy as np # Late on decided to just import it all (got lazy)

%load_ext autoreload
%autoreload 2

## Elliptic function identities

In [3]:
sigma_p_identity = Eq(
    wp(y, g2, g3) - wp(x, g2, g3),
    sigma(x + y, g2, g3) * sigma(x - y, g2, g3) / (sigma(x, g2, g3) ** 2 * sigma(y, g2, g3) ** 2) 
)
pw_to_zw_identity = Eq(
    (wpp(z,g2,g3) - wpp(x,g2,g3))/(wp(z,g2,g3) - wp(x,g2,g3))/2,
    zeta(z + x,g2, g3) - zeta(z,g2, g3) - zeta(x,g2, g3)
)
pw_to_zw_identity_one_sided = Eq(
    wpp(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)),
    2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3)
)

zw_dlog_sigma_identity = Eq(zeta(z - x, g2, g3), Derivative(log(sigma(z - x, g2, g3)), z))
pw_to_dlog_sigma_identity = pw_to_zw_identity.subs([
    zw_dlog_sigma_identity.subs(x,-x).args,
    zw_dlog_sigma_identity.subs(x,0).args,
])
pw_to_dlog_sigma_identity_b = pw_to_dlog_sigma_identity.subs(x,-x).subs([
    (wp(-x,g2,g3), wp(x,g2,g3)), (wpp(-x,g2,g3), -wpp(x,g2,g3)), (zeta(-x, g2,g3), -zeta(x, g2,g3))
])

pw_addition_id = Eq(
    wp(x + y, g2, g3),
    -wp(x, g2, g3) - wp(y, g2, g3) + (wpp(x, g2, g3) - wpp(y, g2, g3))**2/(4*(wp(x, g2, g3) - wp(y, g2, g3))**2)
)

pwp_xy_addition = Eq(wpp(x + y, g2, g3),
    -(wpp(x, g2, g3) + wpp(y, g2, g3))/2 + 
    3*(wp(x, g2, g3) + wp(y, g2, g3))*(wpp(x, g2, g3) - wpp(y, g2, g3))/(2*(wp(x, g2, g3) - wp(y, g2, g3)))
    - (wpp(x, g2, g3) - wpp(y, g2, g3))**3/(4*(wp(x, g2, g3) - wp(y, g2, g3))**3)
)

pwp_sigma_dbl_ratio = Eq(wpp(z,g2,g3), - sigma(2*z,g2,g3)/sigma(z,g2,g3)**4)
pwp_sqrd = Eq(diff(wp(z,g2,g3),z)**2, 4*wp(z,g2,g3)**3 - g2*wp(z,g2,g3) - g3)
pwp_2nd = Eq(
    diff(wp(z,g2,g3),(z,2)),
    solve(Eq(diff(pwp_sqrd.lhs,z), diff(pwp_sqrd.rhs,z)), diff(wp(z,g2,g3),(z,2))
    )[0]
)



# See Homogenity 
# https://functions.wolfram.com/EllipticFunctions/WeierstrassP/introductions/Weierstrass/ShowAll.html
sig_scale_law = Eq(sigma(x,g2*c**4,g3*c**6), sigma(x*c,g2,g3)/c)
zw_scale_law = Eq(zeta(x,g2*c**4,g3*c**6), zeta(x*c,g2,g3)*c)
pw_scale_law = Eq(wp(x,g2*c**4,g3*c**6), wp(x*c,g2,g3)*c**2)
pwp_scale_law = Eq(wpp(x,g2*c**4,g3*c**6), wpp(x*c,g2,g3)*c**3)

omega1_scale_law_a = Eq(omega[1], f(1, g2,g3))
omega1_scale_law_b = Eq(c*omega[1], f(1, g2/c**4,g3/c**6))
omega3_scale_law_a = Eq(omega[3], f(3, g2,g3))
omega3_scale_law_b = Eq(c*omega[3], f(3, g2/c**4,g3/c**6))

quasi_period_sigma_b = Eq(
    sigma(2*m*omega[3] + 2*n*omega[1] + z, g2, g3),
    (-1)**(m*n + m + n)*sigma(z, g2, g3)*
    exp(2*(m*omega[3] + n*omega[1] + z)*zeta(m*omega[3] + n*omega[1], g2, g3))
)

sigma_O2_abt_x = Eq(sigma(x + z, g2, g3), sigma(x, g2, g3) + z*sigma(x, g2, g3)*zeta(x, g2, g3) + O(z**2))


quasi_period_zw = Eq(
    zeta(2*m*omega[3] + 2*n*omega[1] + z, g2, g3), 
    2*eta[1]*n + 2*eta[3]*m + zeta(z, g2, g3)
)
quasi_period_zw_b = Eq(
    zeta(2*m*omega[3] + 2*n*omega[1] + z, g2, g3), 
    2*zeta(omega[1], g2, g3)*n + 2*zeta(omega[3], g2, g3)*m + zeta(z, g2, g3)
)
zw_omega_sum_0 = Eq(
    zeta(omega[1],g2,g3) + zeta(omega[3],g2,g3) - zeta(omega[1] + omega[3],g2,g3),
    0
)
zw_1_2_i_pi_omega = Eq(
    omega[1]*zeta(omega[3],g2,g3) - omega[3]*zeta(omega[1],g2,g3),
    I*pi/2
)
zw_m_n_omega = Eq(
    zeta(m*omega[3] + n*omega[1], g2, g3), 
    m*zeta(omega[3], g2, g3) + n*zeta(omega[1], g2, g3)
)


sigma_p_identity
pw_to_zw_identity
pw_to_zw_identity_one_sided
zw_dlog_sigma_identity
pw_to_dlog_sigma_identity_b
pw_addition_id
pwp_xy_addition
pwp_sigma_dbl_ratio
pwp_sqrd
pwp_2nd

sig_scale_law
zw_scale_law
pw_scale_law
pwp_scale_law
omega1_scale_law_a
omega1_scale_law_b
omega3_scale_law_a
omega3_scale_law_b

quasi_period_sigma_b
sigma_O2_abt_x

quasi_period_zw
quasi_period_zw_b
zw_omega_sum_0
zw_1_2_i_pi_omega
zw_m_n_omega

Eq(-wp(x, g2, g3) + wp(y, g2, g3), sigma(x - y, g2, g3)*sigma(x + y, g2, g3)/(sigma(x, g2, g3)**2*sigma(y, g2, g3)**2))

Eq((-\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), -zeta(x, g2, g3) - zeta(z, g2, g3) + zeta(x + z, g2, g3))

Eq(\wp'(z, g2, g3)/(wp(x, g2, g3) - wp(z, g2, g3)), 2*zeta(z, g2, g3) - zeta(-x + z, g2, g3) - zeta(x + z, g2, g3))

Eq(zeta(-x + z, g2, g3), Derivative(log(sigma(-x + z, g2, g3)), z))

Eq((\wp'(x, g2, g3) + \wp'(z, g2, g3))/(2*(-wp(x, g2, g3) + wp(z, g2, g3))), zeta(x, g2, g3) - Derivative(log(sigma(z, g2, g3)), z) + Derivative(log(sigma(-x + z, g2, g3)), z))

...

Eq(zeta(2*m*omega[3] + 2*n*omega[1] + z, g2, g3), 2*m*eta[3] + 2*n*eta[1] + zeta(z, g2, g3))

Eq(zeta(2*m*omega[3] + 2*n*omega[1] + z, g2, g3), 2*m*zeta(omega[3], g2, g3) + 2*n*zeta(omega[1], g2, g3) + zeta(z, g2, g3))

Eq(-zeta(omega[1] + omega[3], g2, g3) + zeta(omega[1], g2, g3) + zeta(omega[3], g2, g3), 0)

Eq(-zeta(omega[1], g2, g3)*omega[3] + zeta(omega[3], g2, g3)*omega[1], I*pi/2)

Eq(zeta(m*omega[3] + n*omega[1], g2, g3), m*zeta(omega[3], g2, g3) + n*zeta(omega[1], g2, g3))

## The general two mode model with quadratic phase modulation polynomial

In [4]:
N_modes = 2

sum_gamma_j_0 = Eq(Sum(gamma[j], (j, 1, N_modes)), 0)

a_0_eq = Eq(
    a[0],
    - Sum(a[j]*u(z,mu[j])*v(z,mu[j]), (j, 1, N_modes))
    - Sum(a[j,k]/2*u(z,mu[j])*v(z,mu[j])*u(z,mu[k])*v(z,mu[k]), (j, 1, N_modes), (k, 1, N_modes))
    + Product(u(z,mu[j]),(j, 1, N_modes)) + Product(v(z,mu[j]),(j, 1, N_modes))
)

a_0_init = a_0_eq.subs(z,0)

ajk_syms = [(a[j,k], a[k,j]) for j in range(1, N_modes + 1) for k in range(1,  N_modes + 1) if j > k]

Q_poly_uv  = Eq(
    Q(*[u(z, mu[_j + 1])*v(z, mu[_j + 1]) for _j in range(N_modes)]), 
    a[0] + Sum(a[j]*(gamma[j] - rho(z)), (j, 1, N_modes)) + 
    Sum(a[j,k]/2*(gamma[j] - rho(z))*(gamma[k] - rho(z)), (j, 1, N_modes), (k, 1, N_modes))
)
Q_poly_uv_b  = Eq(
    Q_poly_uv.lhs, 
    b[0] + rho(z)*b[1] + rho(z)**2*b[2]
)

# Depend on a[j], a[j,k], gamma[j]
b_j_coeffs = [
    Eq(
        b[0], 
        a[0] + Sum(a[j]*gamma[j], (j, 1, N_modes)) 
        + Sum(a[j,k]/2*gamma[j]*gamma[k], (j, 1, N_modes), (k, 1, N_modes))
    ),
    Eq(
        b[1], 
        -Sum(a[j,k]/2*(gamma[j] + gamma[k]), (j, 1, N_modes), (k, 1, N_modes)) - Sum(a[j], (j, 1, N_modes))
    ),
    Eq(
        b[2], 
        Sum(a[j,k]/2, (j, 1, N_modes), (k, 1, N_modes))
    )
]

assert (
    Q_poly_uv.rhs.doit().expand().collect(rho(z), factor) 
    - Q_poly_uv_b.rhs.subs([_.args for _ in b_j_coeffs]).doit().expand().collect(rho(z), factor)
).simplify() == 0, 'b_j not correctly defined'


_b0_replace = Eq(
    b[0], 
    a[0]
    + (Sum(a[j,j]/4, (j, 1, 2)) - Sum(a[j,k]/8, (j, 1, 2), (k, 1, 2)))*(gamma[1] - gamma[2])**2 
    + Sum(a[j]*gamma[j], (j,1,2))
)
assert (
    (_b0_replace.rhs - b_j_coeffs[0].rhs).doit().subs(ajk_syms).expand().subs(gamma[2], -gamma[1])
).simplify() == 0, '_b0_replace not 0'
b_j_coeffs[0] = _b0_replace

_b1_orig = b_j_coeffs[1]
_ajgamsum = Sum(a[j], (j,1,2)) + Sum(a[j,j]*gamma[j], (j,1,2))
b_j_coeffs[1] = Eq(
    b_j_coeffs[1].lhs,
    b_j_coeffs[1].rhs
    .doit().expand().subs(ajk_syms)
    .collect(a[1,2], factor)
    .subs([sum_gamma_j_0.doit().args])
    + (_ajgamsum.doit() - _ajgamsum)
)

# Depend on gamma[j]
# Also assuming sum over gamma = 0 (normalisation)
c_j_coeffs = [
    Eq(c[0], Product(gamma[j], (j, 1, N_modes))),
    Eq(c[1], 0),
    Eq(c[2], 1)
]
rho_c_prod_to_sum = Eq(Product(-rho(z) + gamma[j], (j, 1, N_modes)), Sum(c[j]*rho(z)**j, (j,0, N_modes)))
assert (
    rho_c_prod_to_sum.lhs.doit().expand().collect(rho(z), factor).subs([sum_gamma_j_0.doit().args]) - 
    rho_c_prod_to_sum.rhs.doit().subs([_.args for _ in c_j_coeffs]).doit()
).simplify() == 0, 'c_j not properly defined'

drhop_b = Eq(diff(rho(z),z)**2, (Q_poly_uv_b.rhs)**2 - 4*Product(gamma[j] - rho(z),(j, 1, N_modes)))
drhop_c = Eq(drhop_b.lhs, drhop_b.rhs.subs(*rho_c_prod_to_sum.args).doit().expand().collect(rho(z), factor))

d_j_coeffs = [Eq(d[_j], drhop_c.doit().rhs.coeff(rho(z),_j)) for _j in range(5)]
drhop_d = Eq(drhop_b.lhs, Sum(d[j]*rho(z)**j, (j, 0, 4)))

assert (
    (drhop_b.rhs.doit() - drhop_d.rhs.doit())
    .subs([_eq.args for _eq in d_j_coeffs])
    .subs([_eq.args for _eq in c_j_coeffs])
    .doit().expand().factor().subs(*sum_gamma_j_0.doit().args)
) == 0, '_rhop_bcd_check failed'

drhopp = Eq(
    diff(drhop_b.lhs,z)/diff(rho(z),z)/2, 
    (diff(drhop_b.rhs.doit(),z)/diff(rho(z),z)/2)
    .subs(b[0] + b[1]*rho(z) + b[2]*rho(z)**2, x)
    .expand().collect(x, factor)
    .subs(x, b[0] + b[1]*rho(z) + b[2]*rho(z)**2)
)

sum_gamma_j_0
a_0_eq
a_0_init
Q_poly_uv
Q_poly_uv_b

for _eq in b_j_coeffs:
    _eq

for _eq in c_j_coeffs:
    _eq

for _eq in d_j_coeffs:
    _eq

rho_c_prod_to_sum

drhop_b
drhop_d
drhopp

Eq(Sum(gamma[j], (j, 1, 2)), 0)

Eq(a[0], Product(u(z, mu[j]), (j, 1, 2)) + Product(v(z, mu[j]), (j, 1, 2)) - Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2)) - Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(a[0], Product(u(0, mu[j]), (j, 1, 2)) + Product(v(0, mu[j]), (j, 1, 2)) - Sum(u(0, mu[j])*v(0, mu[j])*a[j], (j, 1, 2)) - Sum(u(0, mu[j])*u(0, mu[k])*v(0, mu[j])*v(0, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(Q(u(z, mu[1])*v(z, mu[1]), u(z, mu[2])*v(z, mu[2])), a[0] + Sum((-rho(z) + gamma[j])*a[j], (j, 1, 2)) + Sum((-rho(z) + gamma[j])*(-rho(z) + gamma[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(Q(u(z, mu[1])*v(z, mu[1]), u(z, mu[2])*v(z, mu[2])), rho(z)**2*b[2] + rho(z)*b[1] + b[0])

...

Eq(d[4], b[2]**2)

Eq(Product(-rho(z) + gamma[j], (j, 1, 2)), Sum(rho(z)**j*c[j], (j, 0, 2)))

Eq(Derivative(rho(z), z)**2, (rho(z)**2*b[2] + rho(z)*b[1] + b[0])**2 - 4*Product(-rho(z) + gamma[j], (j, 1, 2)))

Eq(Derivative(rho(z), z)**2, Sum(rho(z)**j*d[j], (j, 0, 4)))

Eq(Derivative(rho(z), (z, 2)), (2*rho(z)*b[2] + b[1])*(rho(z)**2*b[2] + rho(z)*b[1] + b[0]) + 2*(-2*rho(z) + gamma[1] + gamma[2]))

The formulas for $b_j$ are the same in the coupler case as they are in the FWM case.

In [5]:

_b0_coupler = Eq(
    b[0],
    (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) 
    + a[0] + Sum(a[j]*gamma[j], (j, 1, 2))
)

_b0_fwm = Eq(
    b[0],
    Sum(gamma[j]*gamma[k]*a[j, k]/2, (j, 1, 2), (k, 1, 2))
    + a[0] + Sum(a[j]*gamma[j], (j, 1, 2))
)

_b1_coupler = Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2)))

_b1_fwm = Eq(b[1], -Sum(a[j, k]*(gamma[j] + gamma[k])/2, (j, 1, 2), (k, 1, 2)) - Sum(a[j], (j, 1, 2)))

_b0_coupler
_b0_fwm

_b1_coupler
_b1_fwm

assert _b0_coupler.doit().subs([(a[2,1],a[1,2]), (gamma[2], -gamma[1])]).expand().subs([
    _b0_fwm.doit().subs([(a[2,1],a[1,2]), (gamma[2], -gamma[1])]).expand().args
]), 'FWM expression (N=4) not equivalent to coupler expression for b0 (N=2)'
assert _b1_coupler.doit().subs([(a[2,1],a[1,2]), (gamma[2], -gamma[1])]).expand().subs([
    _b1_fwm.doit().subs([(a[2,1],a[1,2]), (gamma[2], -gamma[1])]).expand().args
]), 'FWM expression (N=4) not equivalent to coupler expression for b0 (N=2)'

Eq(b[0], (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) + a[0] + Sum(a[j]*gamma[j], (j, 1, 2)))

Eq(b[0], a[0] + Sum(a[j]*gamma[j], (j, 1, 2)) + Sum(a[j, k]*gamma[j]*gamma[k]/2, (j, 1, 2), (k, 1, 2)))

Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2)))

Eq(b[1], -Sum(a[j], (j, 1, 2)) - Sum((gamma[j] + gamma[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)))

In [6]:
gamma12_ids = [
    sum_gamma_j_0.doit(),
#     Eq(
#         _gam1term.expand(),
#         _gam1term.subs([sum_gamma_j_0.doit().args])
#     ),
    Eq(
        (gamma[1] + gamma[2] + gamma[1] - gamma[2] )/2,
        (gamma[1] - gamma[2] )/2
    ),
    Eq(
        (gamma[1] + gamma[2] - gamma[1] + gamma[2] )/2,
        (-gamma[1] + gamma[2] )/2
    )
]

gammajk_trace_id = Eq(
    Sum((gamma[j] + gamma[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)),
    Sum(a[j, j]*gamma[j], (j, 1, 2))
)
assert (
    (gammajk_trace_id.rhs - gammajk_trace_id.lhs).doit().subs([sum_gamma_j_0.doit().args]) == 0
), 'gammajk_trace_id not true'


for _ in gamma12_ids:
    _
    
gammajk_trace_id

Eq(gamma[1] + gamma[2], 0)

Eq(gamma[1], gamma[1]/2 - gamma[2]/2)

Eq(gamma[2], -gamma[1]/2 + gamma[2]/2)

Eq(Sum((gamma[j] + gamma[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)), Sum(a[j, j]*gamma[j], (j, 1, 2)))

## Equations of motion

In [7]:
u_js = [u(z, mu[_j+1]) for _j in range(N_modes)]
v_js = [v(z, mu[_j+1]) for _j in range(N_modes)]

du_js = [diff(_, z) for _ in u_js]
dv_js = [diff(_, z) for _ in v_js]

u_v_js = u_js + v_js
d_u_v_js = du_js + dv_js

hamiltonian_a0 = Eq(H(*u_v_js), a[0])
hamiltonian_u_v = hamiltonian_a0.subs([a_0_eq.args])

u_eoms = [
    Eq(
        du_js[_j], 
        diff(hamiltonian_u_v.rhs.doit(), v_js[_j]).subs(ajk_syms)
    )
    for _j in range(N_modes)
]
v_eoms = [
    Eq(
        dv_js[_j], 
        -diff(hamiltonian_u_v.rhs.doit(), u_js[_j]).subs(ajk_syms)
    )
    for _j in range(N_modes)
]

_a_jks = [a[_j + 1,_k + 1] for _j in range(N_modes) for _k in range(N_modes)]
_a_js = [a[_j + 1] for _j in range(N_modes)]
_no_ajs_ajks = [(_, 0) for _ in _a_js + _a_jks]
u_eoms = [Eq(_.lhs, (_.rhs -_.rhs.subs(_no_ajs_ajks)).factor() + _.rhs.subs(_no_ajs_ajks)) for _ in u_eoms]
v_eoms = [Eq(_.lhs, (_.rhs -_.rhs.subs(_no_ajs_ajks)).factor() + _.rhs.subs(_no_ajs_ajks)) for _ in v_eoms]

u_v_eoms = u_eoms + v_eoms
u_v_eoms_subs = [_.args for _ in u_v_eoms]

assert (
    diff(hamiltonian_u_v.rhs.doit().subs(ajk_syms),z).subs(u_v_eoms_subs).simplify() == 0
), 'Hamiltonian not conserved'


duv_js = [
    Eq(
        Derivative(u(z, mu[j])*v(z, mu[j]), z)
        .subs(j, _j + 1), 
        diff(u(z, mu[j])*v(z, mu[j]), z).doit()
        .subs(j, _j + 1)
        .subs(u_v_eoms_subs).simplify()
    )
    for _j in range(2)
]

duvj = Eq(
    Derivative(u(z, mu[j])*v(z, mu[j]), z), 
    - Product(u(z, mu[j]), (j, 1, N_modes)) 
    + Product(v(z, mu[j]), (j, 1, N_modes))
)

assert not any([
    (duvj.lhs - duvj.rhs).subs([(N_modes, 2), (j, _j + 1)] + [_.args for _ in duv_js]).doit() != 0
    for _j in range(2)
]), 'duvj badly defined'


hamiltonian_a0
hamiltonian_u_v
for _ in u_v_eoms:
    _
print()
duvj
for _ in duv_js:
    _

Eq(H(u(z, mu[1]), u(z, mu[2]), v(z, mu[1]), v(z, mu[2])), a[0])

Eq(H(u(z, mu[1]), u(z, mu[2]), v(z, mu[1]), v(z, mu[2])), Product(u(z, mu[j]), (j, 1, 2)) + Product(v(z, mu[j]), (j, 1, 2)) - Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2)) - Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(Derivative(u(z, mu[1]), z), -(u(z, mu[1])*v(z, mu[1])*a[1, 1] + u(z, mu[2])*v(z, mu[2])*a[1, 2] + a[1])*u(z, mu[1]) + v(z, mu[2]))

Eq(Derivative(u(z, mu[2]), z), -(u(z, mu[1])*v(z, mu[1])*a[1, 2] + u(z, mu[2])*v(z, mu[2])*a[2, 2] + a[2])*u(z, mu[2]) + v(z, mu[1]))

Eq(Derivative(v(z, mu[1]), z), (u(z, mu[1])*v(z, mu[1])*a[1, 1] + u(z, mu[2])*v(z, mu[2])*a[1, 2] + a[1])*v(z, mu[1]) - u(z, mu[2]))

Eq(Derivative(v(z, mu[2]), z), (u(z, mu[1])*v(z, mu[1])*a[1, 2] + u(z, mu[2])*v(z, mu[2])*a[2, 2] + a[2])*v(z, mu[2]) - u(z, mu[1]))

Eq(Derivative(u(z, mu[j])*v(z, mu[j]), z), -Product(u(z, mu[j]), (j, 1, 2)) + Product(v(z, mu[j]), (j, 1, 2)))

Eq(Derivative(u(z, mu[1])*v(z, mu[1]), z), -u(z, mu[1])*u(z, mu[2]) + v(z, mu[1])*v(z, mu[2]))

Eq(Derivative(u(z, mu[2])*v(z, mu[2]), z), -u(z, mu[1])*u(z, mu[2]) + v(z, mu[1])*v(z, mu[2]))

### Solving for $\rho(z)$

This part follows the four-wave mixing case very closely. Both equations involve a quartic in $\rho$.

We now solve for $\rho(z)$ in terms of the Weierstrass $\wp$ elliptic function:

In [8]:
rho_z_mean_uv = Eq(rho(z), -Sum(u(z,mu[j])*v(z,mu[j]), (j,1, N_modes))/N_modes)

rho_lam = Eq(diff(rho(z),z)**2, d[4]*Product(rho(z) - lamda[j],(j, 1, 4)))
rho_q_lam1 = Eq(rho(z), q(z) + lamda[1])
qz_lam = rho_lam.subs(*rho_q_lam1.args).doit()


# -- q to s transform --

q_s = Eq(q(z), 1/s(z))
sz_lam = qz_lam.subs(*q_s.args).doit()
Omega_lams = [Eq(Omega[_j], -1/(lamda[1] - lamda[_j+1])) for _j in range(1,4)]
sz_lam_b = Eq(
    Derivative(s(z), z)**2,
    -(lamda[1] - lamda[2])*(lamda[1] - lamda[3])*(lamda[1] - lamda[4])*
    (s(z) - 1/(lamda[1] - lamda[2]))*(s(z) - 1/(lamda[1] - lamda[3]))*(s(z) - 1/(lamda[1] - lamda[4]))*d[4]
)
sz_lam_c = Eq(
    Derivative(s(z), z)**2, 
    -Product(s(z) - Omega[j],(j,1,3))*d[4]/Product(Omega[j],(j,1,3))
)

_q_to_s_test = rho_lam.subs([
    rho_q_lam1.args,
    qz_lam.args,
    q_s.args
]).doit()
_q_to_s_test = Eq(_q_to_s_test.lhs*s(z)**4, _q_to_s_test.rhs*s(z)**4)
_q_to_s_test_lhs = _q_to_s_test.lhs - sz_lam_c.lhs
_q_to_s_test_rhs = (
    _q_to_s_test.rhs - sz_lam_c.rhs.doit().subs([_eq.args for _eq in Omega_lams])
).expand().simplify()
if not _q_to_s_test_lhs == 0:
    raise ValueError('_q_to_s_test_lhs failed')
if not _q_to_s_test_rhs == 0:
    raise ValueError('_q_to_s_test_lhs failed')
    
    
# -- s to w transform -- 

sz_w_Omega = Eq(s(z), -4*Product(Omega[l],(l,1,3))/d[4]*w(z) + Sum(Omega[l]/3,(l,1,3)))

g2_Omega = Eq(
    g2, 
    d[4]**2*Sum((Omega[j] - Omega[k])**2, (j,1,3), (k,1,3))/48/Product(Omega[j],(j,1,3))**2
)
g3_Omega = Eq(
    g3, 
    -(Product(Omega[j] - Sum(Omega[k]/3,(k,1,3)),(j,1,3))*d[4]**3)/16/Product(Omega[j],(j,1,3))**3
)

dw_g2_g3 = Eq(diff(w(z),z)**2, 4*w(z)**3 - g2*w(z) - g3)
wz_wp_sol = Eq(w(z), wp(z-z0, g2, g3))

_sw_check = (
    sz_lam_c
    .subs(*sz_w_Omega.args).doit()
    .subs(*dw_g2_g3.subs([g2_Omega.args, g3_Omega.args]).args)
    .simplify()
)
if not _sw_check:
    raise ValueError('_sw_check failed')
    
    
## -- define z1 -- 

rho_wp = rho_q_lam1.subs([q_s.args, sz_w_Omega.args, wz_wp_sol.args])

dw_xi = Eq(diff(w(z),z)**2, 4*Product(w(z) - xi[j], (j,1,3)))
xi_j_omega = Eq(xi[j], (Sum(Omega[l]/3, (l, 1, 3)) - Omega[j])/(4*Product(Omega[l], (l, 1, 3))/d[4]))
xi_j_omega_sum = Eq(Sum(xi[j], (j,1,3)), Sum(xi[j].subs(*xi_j_omega.args), (j,1,3)).doit().simplify())
xi_j_omega_sum_b = Eq(xi[j]*(4*Product(Omega[l], (l, 1, 3)))/d[4] + Omega[j], Sum(Omega[l]/3, (l, 1, 3)))
xi_j_root = Eq(xi[j], wp(omega[j], g2, g3))

wp_z1 = Eq(wp(z1, g2, g3), Sum(Omega[l]/3, (l, 1, 3))/(4*Product(Omega[l], (l, 1, 3))/b[2]**2))
wpp_z1 = Eq(wpp(z1, g2, g3), b[2]**3/(4*Product(Omega[l], (l, 1, 3))))

wpp_z1_sqrd = Eq(
    wpp(z1,g2,g3)**2, 
    (
        (4*wp(z1, g2, g3)**3 - g2*wp(z1, g2, g3) - g3)
        .subs([
            wp_z1.args,
            g2_Omega.args,
            g3_Omega.args
        ]).expand().doit().expand()
        - b[2]**6/16/Product(Omega[j],(j,1,3))**2 
    ).simplify()
    + b[2]**6/16/Product(Omega[j],(j,1,3))**2
).subs([(Product(Omega[j], (j, 1, 3)), Product(Omega[l], (l, 1, 3))), (d[4], b[2]**2)])

z1_wpp_sqrd_check = Eq(wpp_z1.lhs**2 - wpp_z1_sqrd.lhs, wpp_z1.rhs**2 - wpp_z1_sqrd.rhs)
if not z1_wpp_sqrd_check:
    raise ValueError('z1_wpp_sqrd_check failed')
    
## -- 
    
rho_z_sqrt_d4 = Eq(
    rho(z),
    lamda[1] + 
    wpp_z1.lhs/wpp_z1.rhs/(
        -wp(z - z0, g2, g3) + 
        Sum(Omega[l]/3, (l, 1, 3))/(4*Product(Omega[l], (l, 1, 3))/b[2]**2)
    ).subs(wp_z1.rhs, wp_z1.lhs)/(4*Product(Omega[l], (l, 1, 3))/b[2]**2)
)

rho_z_sqrt_d4_test = Eq(
    rho_z_sqrt_d4.subs([wp_z1.args]).lhs,
    (rho_z_sqrt_d4.subs([wp_z1.args]).rhs - lamda[1])*wpp_z1.rhs/wpp_z1.lhs + lamda[1]
)
rho_z_sqrt_d4_check_lhs = (rho_wp.lhs - rho_z_sqrt_d4_test.lhs).simplify() == 0
rho_z_sqrt_d4_check_rhs = (rho_wp.rhs - rho_z_sqrt_d4_test.rhs).subs(d[4], b[2]**2).simplify() == 0

if not rho_z_sqrt_d4_check_lhs:
    raise ValueError('rho_z_sqrt_d4_check_lhs failed')
if not rho_z_sqrt_d4_check_rhs:
    raise ValueError('rho_z_sqrt_d4_check_rhs failed')


## -- zeta function expansion --

zeta_z_z0_z1 = pw_to_zw_identity_one_sided.subs([(z,t),(x,y)]).subs([(t, z1), (y, z - z0)])
rho_zeta = Eq(
    rho_z_sqrt_d4.lhs, 
    (rho_z_sqrt_d4.rhs - lamda[1] + zeta_z_z0_z1.lhs/b[2]).simplify() + 
    lamda[1] - zeta_z_z0_z1.rhs/b[2]
)

## -- Derivative of rho as wp -- 

drho_zeta_d4 = Eq(diff(rho_zeta.lhs,z), diff(rho_zeta.rhs,z).expand())
drho_wp_d4 = Eq(
    Derivative(rho(z), z),
    -(wp(z - z0 + z1, g2, g3) - wp(z - z0 - z1,g2,g3))/b[2]
)
drho_wp_d4_b = Eq(
    diff(rho_z_sqrt_d4.lhs, z), 
    diff(rho_z_sqrt_d4.rhs, z).subs(diff(wp(z-z0,g2,g3),z), wpp(z-z0,g2,g3))
)
# Note this is true for all z,z1
drho_wp_id = Eq(drho_wp_d4.lhs/drho_wp_d4_b.lhs, drho_wp_d4.rhs/drho_wp_d4_b.rhs).subs(z,z+z0)

# Important function which appears in denominators
wp_z1_lam_uv_j = rho_z_sqrt_d4.subs([rho_z_mean_uv.args])
wp_z1_lam_uv_j = Eq(wp_z1_lam_uv_j.rhs - lamda[1], wp_z1_lam_uv_j.lhs - lamda[1])


drho_wp_d4_2nd = Eq(
    diff(drho_wp_d4.lhs,z), 
    diff(drho_wp_d4.rhs,z).replace(diff(wp(wild,g2,g3),wild), wpp(wild, g2, g3)).doit()
)
drho_wp_d4_2nd_b = Eq(
    diff(drho_wp_d4_b.lhs,z), 
    diff(drho_wp_d4_b.rhs,z)
    .replace(diff(wp(wild,g2,g3),wild), wpp(wild, g2, g3)).doit()
    .replace(diff(wpp(wild, g2, g3),wild), diff(wp(wild, g2, g3),(wild, 2)))
    .replace(*pwp_2nd.subs(z, wild).args).doit()
)

drho_wp_d4_2nd_c = Eq(
    (pwp_xy_addition.lhs.subs([(x, z-z0),(y,-z1)]) - pwp_xy_addition.lhs.subs([(x, z-z0),(y,z1)])),
    (pwp_xy_addition.rhs.subs([(x, z-z0),(y,-z1)]) - pwp_xy_addition.rhs.subs([(x, z-z0),(y,z1)]))
    .subs([(wp(-z1,g2,g3), wp(z1,g2,g3)), (wpp(-z1,g2,g3), -wpp(z1,g2,g3))])
)
drho_wp_d4_2nd_c = Eq(
    drho_wp_d4_2nd_c.lhs, 
    sum(
        _.expand().collect(x, factor) for _ in
        apart(drho_wp_d4_2nd_c.rhs.expand().simplify().factor().subs(wp(z-z0,g2,g3),x + wp(z1,g2,g3)),x).args
       ).subs(x, wp(z-z0,g2,g3) - wp(z1,g2,g3))
)
drho_wp_d4_2nd_c = Eq(drho_wp_d4_2nd_c.lhs/b[2], sum(_/b[2] for _ in drho_wp_d4_2nd_c.rhs.args))
drho_wp_d4_2nd_c = drho_wp_d4_2nd.subs([drho_wp_d4_2nd_c.args])


rho_lam
rho_q_lam1
qz_lam
q_s

for _eq in Omega_lams:
    _eq

sz_lam_c

sz_w_Omega
g2_Omega
g3_Omega
dw_g2_g3
wz_wp_sol
rho_wp
xi_j_omega
xi_j_omega_sum
xi_j_omega_sum_b
xi_j_root

wp_z1
wpp_z1
rho_z_sqrt_d4
rho_zeta

drho_wp_d4
drho_wp_d4_b
drho_wp_d4_2nd
drho_wp_d4_2nd_b
drho_wp_d4_2nd_c
drho_wp_id

wp_z1_lam_uv_j

Eq(Derivative(rho(z), z)**2, d[4]*Product(rho(z) - lamda[j], (j, 1, 4)))

Eq(rho(z), q(z) + lamda[1])

Eq(Derivative(q(z), z)**2, (q(z) + lamda[1] - lamda[2])*(q(z) + lamda[1] - lamda[3])*(q(z) + lamda[1] - lamda[4])*q(z)*d[4])

Eq(q(z), 1/s(z))

Eq(Omega[1], -1/(lamda[1] - lamda[2]))

...

Eq(Derivative(rho(z), (z, 2)), (\wp'(z - z0 - z1, g2, g3) - \wp'(z - z0 + z1, g2, g3))/b[2])

Eq(Derivative(rho(z), (z, 2)), (-g2/2 + 6*wp(z - z0, g2, g3)**2)*\wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))**2*b[2]) + 2*\wp'(z1, g2, g3)*\wp'(z - z0, g2, g3)**2/((wp(z1, g2, g3) - wp(z - z0, g2, g3))**3*b[2]))

Eq(Derivative(rho(z), (z, 2)), -(\wp'(z1, g2, g3)**2 + 3*\wp'(z - z0, g2, g3)**2)*\wp'(z1, g2, g3)/(2*(-wp(z1, g2, g3) + wp(z - z0, g2, g3))**3*b[2]) + 4*\wp'(z1, g2, g3)/b[2] + 6*\wp'(z1, g2, g3)*wp(z1, g2, g3)/((-wp(z1, g2, g3) + wp(z - z0, g2, g3))*b[2]))

Eq(1, (-wp(z, g2, g3) + wp(z1, g2, g3))**2*(wp(z - z1, g2, g3) - wp(z + z1, g2, g3))/(\wp'(z, g2, g3)*\wp'(z1, g2, g3)))

Eq(\wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*b[2]), -lamda[1] - Sum(u(z, mu[j])*v(z, mu[j]), (j, 1, 2))/2)

The zeros of $\rho'(z)$ are thus the four points:

In [9]:
rhop_zeros = [
    Eq(rhop(z0),0),
    Eq(rhop(z0+omega[1]),0),
    Eq(rhop(z0+omega[2]),0),
    Eq(rhop(z0+omega[3]),0)
]
for _eq in rhop_zeros:
    _eq

Eq(\rho'(z0), 0)

Eq(\rho'(z0 + omega[1]), 0)

Eq(\rho'(z0 + omega[2]), 0)

Eq(\rho'(z0 + omega[3]), 0)

The modal power can also be written as follows:

In [10]:
uv_j_rho = Eq(u(z,mu[j])*v(z,mu[j]), gamma[j] - rho(z))
gamma_uv_0 = Eq(gamma[j], solve(uv_j_rho,gamma[j])[0].subs(z,0))
rho_z0_lam1 = Eq(rho(z0), lamda[1])
rho0_lam_1 = rho_z_sqrt_d4.subs(z,0).subs([gamma_uv_0.args]).subs(wp(-z0,g2,g3), wp(z0,g2,g3))
gamma_j_lam_1 = gamma_uv_0.subs([rho0_lam_1.args])
uv_uv_0_j = uv_j_rho.subs([
    gamma_uv_0.args, 
    rho_z_sqrt_d4.args, 
    rho_z_sqrt_d4.subs(z,0).args,
    (wp(-z0,g2,g3), wp(z0,g2,g3))
])
uv_mu_j_z1p = Eq(
    uv_uv_0_j.lhs - uv_uv_0_j.subs(z, mu[j]).subs(u(mu[j], mu[j]),0).lhs, 
    uv_uv_0_j.rhs - uv_uv_0_j.subs(z, mu[j]).subs(u(mu[j], mu[j]),0).rhs
)

uv_j_rho
rho_z0_lam1
rho0_lam_1
gamma_j_lam_1
uv_uv_0_j
uv_mu_j_z1p

Eq(u(z, mu[j])*v(z, mu[j]), -rho(z) + gamma[j])

Eq(rho(z0), lamda[1])

Eq(rho(0), lamda[1] + \wp'(z1, g2, g3)/((-wp(z0, g2, g3) + wp(z1, g2, g3))*b[2]))

Eq(gamma[j], u(0, mu[j])*v(0, mu[j]) + lamda[1] + \wp'(z1, g2, g3)/((-wp(z0, g2, g3) + wp(z1, g2, g3))*b[2]))

Eq(u(z, mu[j])*v(z, mu[j]), u(0, mu[j])*v(0, mu[j]) - \wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*b[2]) + \wp'(z1, g2, g3)/((-wp(z0, g2, g3) + wp(z1, g2, g3))*b[2]))

Eq(u(z, mu[j])*v(z, mu[j]), \wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*b[2]) - \wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*b[2]))

The invariants $g_2, g_3$ can be expressed in terms of the parameters via substitution of symmetric polynomials in $\Omega_j$:

In [11]:
lam_Omegas = [Eq(lamda[_j+2], solve(Omega_lams[_j],lamda[_j+2])[0]).args for _j in range(3)]
qz_Omega = qz_lam.subs(lam_Omegas)
qz_Omega_b = Eq(qz_Omega.lhs, qz_Omega.rhs.expand().collect(q(z), factor))
dj_Omega = [
    Eq(
        qz_Omega_b.rhs.expand().coeff(q(z),_j).factor(), 
        drhop_d.subs([rho_q_lam1.args]).doit().rhs.expand().coeff(q(z), _j)
    ) for _j in range(4)
]

g2_dj = Eq(
    (
        g2_Omega.lhs - g2_Omega.rhs.doit().expand().simplify().doit().expand().simplify() +
        (dj_Omega[2].lhs**2/12 - dj_Omega[1].lhs*dj_Omega[3].lhs/4).expand().simplify() +
        dj_Omega[0].lhs*d[4]
    ).expand().simplify(),
    (
        (dj_Omega[2].rhs**2/12 - dj_Omega[1].rhs*dj_Omega[3].rhs/4).expand().simplify() +
        dj_Omega[0].rhs*d[4]
    ).expand().simplify()
)

g3_dj = Eq(
    (
        g3_Omega.lhs - g3_Omega.rhs.doit().expand().simplify().doit().expand().simplify() -
        (
            dj_Omega[2].lhs**3/216 - 
            dj_Omega[2].lhs*dj_Omega[3].lhs*dj_Omega[1].lhs/48  + 
            d[4]*dj_Omega[1].lhs**2/16 -
            dj_Omega[0].lhs*(8*d[2]*d[4] - 3*d[3]**2)/48
        )
    ).expand().simplify(),
    - (
        dj_Omega[2].rhs**3/216 - 
        dj_Omega[2].rhs*dj_Omega[3].rhs*dj_Omega[1].rhs/48  + 
        d[4]*dj_Omega[1].rhs**2/16 -
        dj_Omega[0].rhs*(8*d[2]*d[4] - 3*d[3]**2)/48
    ).expand().simplify()
)
g2_bcj = g2_dj.subs([_eq.args for _eq in d_j_coeffs]).expand().simplify()
g3_bcj = g3_dj.subs([_eq.args for _eq in d_j_coeffs]).expand().simplify()
Delta_g = Eq(Delta(g2,g3), (g2**3 - 27*g3**2).subs([g2_dj.args, g3_dj.args]).expand().factor())

qz_Omega
qz_Omega_b
for _eq in dj_Omega:
    _eq

g2_dj
g3_dj
Delta_g

# g2_bcj
# g3_bcj

Eq(Derivative(q(z), z)**2, (q(z) - 1/Omega[1])*(q(z) - 1/Omega[2])*(q(z) - 1/Omega[3])*q(z)*d[4])

Eq(Derivative(q(z), z)**2, -(Omega[1]*Omega[2] + Omega[1]*Omega[3] + Omega[2]*Omega[3])*q(z)**3*d[4]/(Omega[1]*Omega[2]*Omega[3]) + (Omega[1] + Omega[2] + Omega[3])*q(z)**2*d[4]/(Omega[1]*Omega[2]*Omega[3]) + q(z)**4*d[4] - q(z)*d[4]/(Omega[1]*Omega[2]*Omega[3]))

Eq(0, d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3 + d[4]*lamda[1]**4)

Eq(-d[4]/(Omega[1]*Omega[2]*Omega[3]), d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)

Eq((Omega[1] + Omega[2] + Omega[3])*d[4]/(Omega[1]*Omega[2]*Omega[3]), d[2] + 3*d[3]*lamda[1] + 6*d[4]*lamda[1]**2)

Eq(-(Omega[1]*Omega[2] + Omega[1]*Omega[3] + Omega[2]*Omega[3])*d[4]/(Omega[1]*Omega[2]*Omega[3]), d[3] + 4*d[4]*lamda[1])

Eq(g2, d[0]*d[4] - d[1]*d[3]/4 + d[2]**2/12)

Eq(g3, d[0]*d[2]*d[4]/6 - d[0]*d[3]**2/16 - d[1]**2*d[4]/16 + d[1]*d[2]*d[3]/48 - d[2]**3/216)

Eq(Delta(g2, g3), (256*d[0]**3*d[4]**3 - 192*d[0]**2*d[1]*d[3]*d[4]**2 - 128*d[0]**2*d[2]**2*d[4]**2 + 144*d[0]**2*d[2]*d[3]**2*d[4] - 27*d[0]**2*d[3]**4 + 144*d[0]*d[1]**2*d[2]*d[4]**2 - 6*d[0]*d[1]**2*d[3]**2*d[4] - 80*d[0]*d[1]*d[2]**2*d[3]*d[4] + 18*d[0]*d[1]*d[2]*d[3]**3 + 16*d[0]*d[2]**4*d[4] - 4*d[0]*d[2]**3*d[3]**2 - 27*d[1]**4*d[4]**2 + 18*d[1]**3*d[2]*d[3]*d[4] - 4*d[1]**3*d[3]**3 - 4*d[1]**2*d[2]**3*d[4] + d[1]**2*d[2]**2*d[3]**2)/256)

The point $z_1$ is then defined as:

In [12]:
one_over_Omega_j_w_wp = Eq(
    1/(Omega[j]),
    wpp(z1, g2, g3)/(wp(z1, g2, g3) - wp(omega[j], g2, g3))/b[2]
)

one_over_Omega_j_w_wp_check_eq = one_over_Omega_j_w_wp.subs([(xi_j_root.rhs, xi_j_root.lhs), wp_z1.args])
one_over_Omega_j_w_wp_check_val = (
    one_over_Omega_j_w_wp_check_eq.lhs.subs(Omega[j], solve(xi_j_omega_sum_b, Omega[j])[0]) - 
    one_over_Omega_j_w_wp_check_eq.rhs*wpp_z1.rhs/wpp_z1.lhs
).doit().subs(d[4], b[2]**2).simplify()
one_over_Omega_j_w_wp_check = one_over_Omega_j_w_wp_check_val == 0
if not one_over_Omega_j_w_wp_check:
    raise ValueError('one_over_Omega_j_w_wp_check failed')
    
Omega_j_w_wp = Eq(1/one_over_Omega_j_w_wp.lhs, 1/one_over_Omega_j_w_wp.rhs)
    
Omega_j_subs = [Omega_j_w_wp.subs(j,_j+1).args for _j in range(3)]
wp_omega_sum = Eq(wp(omega[1], g2, g3) + wp(omega[2], g2, g3) + wp(omega[3], g2, g3), 0)
wp_omega_z1_prod = Eq(
    (wp(z1, g2, g3) - wp(omega[1], g2, g3))*
    (wp(z1, g2, g3) - wp(omega[2], g2, g3))*
    (wp(z1, g2, g3) - wp(omega[3], g2, g3)),
    (wpp(z1,g2,g3)**2)/4
)
wp_omega_g2 = Eq(
    wp(omega[1], g2, g3)*wp(omega[2], g2, g3) + 
    wp(omega[1], g2, g3)*wp(omega[3], g2, g3) + 
    wp(omega[2], g2, g3)*wp(omega[3], g2, g3),
    -g2/4
)

d_j_coeffs_wp = [
    _eq.subs(Omega_j_subs).simplify()
    .subs([wp_omega_sum.args, wp_omega_z1_prod.args]) 
    .expand().simplify()
    .subs([wp_omega_g2.args])
    .subs(
        2*(wp_omega_sum.lhs*wp(z1,g2,g3)).expand(), 
        2*(wp_omega_sum.rhs*wp(z1,g2,g3)).expand()
    )
    for _eq in dj_Omega
]

one_over_Omega_j_w_wp
for _eq in d_j_coeffs_wp:
    _eq


Eq(1/Omega[j], \wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(omega[j], g2, g3))*b[2]))

Eq(d[0] + d[1]*lamda[1] + d[2]*lamda[1]**2 + d[3]*lamda[1]**3 + d[4]*lamda[1]**4, 0)

Eq(4*\wp'(z1, g2, g3)*d[4]/b[2]**3, -d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)

Eq(12*wp(z1, g2, g3)*d[4]/b[2]**2, d[2] + 3*d[3]*lamda[1] + 6*d[4]*lamda[1]**2)

Eq(d[3] + 4*d[4]*lamda[1], 4*(g2/4 - 3*wp(z1, g2, g3)**2)*d[4]/(\wp'(z1, g2, g3)*b[2]))

In [13]:
wp_z1_dj = Eq(wp(z1,g2,g3), solve(d_j_coeffs_wp[2], wp(z1,g2,g3))[0].subs(b[2]**2, d[4]))
wpp_z1_dj = Eq(wpp(z1,g2,g3), solve(d_j_coeffs_wp[1], wpp(z1,g2,g3))[0].subs(b[2]**3, d[4]*b[2]))

wp_z1_dj
wpp_z1_dj

Eq(wp(z1, g2, g3), d[2]/12 + d[3]*lamda[1]/4 + d[4]*lamda[1]**2/2)

Eq(\wp'(z1, g2, g3), (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)*b[2]/4)

the point $z_0$ is defined as:

In [14]:
rho0_lam_1 = rho_z_sqrt_d4.subs(z,0).subs([gamma_uv_0.args]).subs(wp(-z0,g2,g3), wp(z0,g2,g3))
z1_wp_subs = [
    (wp(z1,g2,g3), solve(d_j_coeffs_wp[2], wp(z1,g2,g3))[0]),
    (wpp(z1,g2,g3), solve(d_j_coeffs_wp[1], wpp(z1,g2,g3))[0]),
    (b[2]**2, d[4])
]
wp_z0_rho0_lam1 = Eq(
    wp(z0,g2,g3), 
    solve(rho0_lam_1, wp(z0,g2,g3))[0].expand().collect(wp(z1,g2,g3), factor)
    .subs(z1_wp_subs)
)
droh_0_z0 = Eq(
    diff(rho_z_sqrt_d4.lhs,z).subs(z,0).simplify(), 
    diff(rho_z_sqrt_d4.rhs,z).subs(z,0).simplify().subs(wp(-z0,g2,g3),wp(z0,g2,g3))*
    (-wpp(z0,g2,g3)/diff(wp(z-z0,g2,g3),z).subs(z,0).doit())
)
wpp_z0_rho0_lam1 = Eq(
    wpp(z0,g2,g3), 
    solve(droh_0_z0.subs(*wp_z0_rho0_lam1.args).subs(z1_wp_subs), wpp(z0,g2,g3))[0]
)


wp_z0_rho0_lam1
wpp_z0_rho0_lam1

Eq(wp(z0, g2, g3), d[2]/12 + d[3]*lamda[1]/4 + d[4]*lamda[1]**2/2 + (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*(-rho(0) + lamda[1])))

Eq(\wp'(z0, g2, g3), (d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)*Subs(Derivative(rho(z), z), z, 0)/(4*(rho(0) - lamda[1])**2))

and the point $-z_0+\mu_j$ is given by:

In [15]:
rho_mu_j_gamma = Eq(rho(mu[j]), gamma[j])
wp_z0_mu_j_d = Eq(
    wp(mu[j]-z0,g2,g3), 
    solve(rho_z_sqrt_d4.subs([(z, mu[j]), rho_mu_j_gamma.args]), wp(mu[j]-z0,g2,g3))[0]
    .expand().collect(b[2], factor)
    .subs([
        (wp(z1,g2,g3), solve(d_j_coeffs_wp[2], wp(z1,g2,g3))[0]),
        (wpp(z1,g2,g3), solve(d_j_coeffs_wp[1], wpp(z1,g2,g3))[0]),
        (b[2]**2, d[4])
    ])
)
rhop_mu_j_gamma = Eq(
    rhop(mu[j]), 
    Q(gamma[1] - gamma[j], gamma[2] - gamma[j])
)
rhop_mu_j_gamma_b = rhop_mu_j_gamma.subs([
    Q_poly_uv_b
    .subs([(u(z, mu[_j])*v(z, mu[_j]), gamma[_j] - rho(z)) for _j in range(1,3)])
    .subs(rho(z), gamma[j]).args,
    
])
drho_wp_d4_b_at_mu_j = (
    drho_wp_d4_b.subs(z, mu[j])
    .subs(diff(rho(mu[j]),mu[j]), rhop(mu[j]))
    .subs(z1_wp_subs)
    .subs(*wp_z0_mu_j_d.args)
    .subs(*rhop_mu_j_gamma_b.args)
)
wpp_z0_mu_j_d = Eq(wpp(mu[j]-z0,g2,g3), solve(drho_wp_d4_b_at_mu_j, wpp(mu[j]-z0,g2,g3))[0].factor())

# check the rhop(mu_j) sign works by comparing with wpp
rhop_muj_sign_check = Eq(
    wpp(mu[j]-z0,g2,g3),
    solve(drho_wp_d4_b.subs([
        (diff(rho(z),z), rhop(z)),
        (z, mu[j]),
        wp_z0_mu_j_d.args,
        wp_z1_dj.args,
        wpp_z1_dj.args,
        rhop_mu_j_gamma_b.args
    ]), wpp(mu[j]-z0,g2,g3))[0].factor()
)

rhop_muj_sign_check_lhs = (wpp_z0_mu_j_d.lhs - rhop_muj_sign_check.lhs).simplify() == 0
rhop_muj_sign_check_rhs = (wpp_z0_mu_j_d.rhs - rhop_muj_sign_check.rhs).simplify() == 0

if not rhop_muj_sign_check_lhs:
    raise ValueError('rhop_muj_sign_check_lhs failed')
if not rhop_muj_sign_check_rhs:
    raise ValueError('rhop_muj_sign_check_rhs failed')
    


rho_mu_j_gamma
wp_z0_mu_j_d

drho_wp_d4_b
rhop_mu_j_gamma
rhop_mu_j_gamma_b
wpp_z0_mu_j_d 

Eq(rho(mu[j]), gamma[j])

Eq(wp(-z0 + mu[j], g2, g3), d[2]/12 + d[3]*lamda[1]/4 + d[4]*lamda[1]**2/2 - (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1])))

Eq(Derivative(rho(z), z), \wp'(z1, g2, g3)*\wp'(z - z0, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))**2*b[2]))

Eq(\rho'(mu[j]), Q(gamma[1] - gamma[j], gamma[2] - gamma[j]))

Eq(\rho'(mu[j]), b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)

Eq(\wp'(-z0 + mu[j], g2, g3), -(b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)*(d[1] + 2*d[2]*lamda[1] + 3*d[3]*lamda[1]**2 + 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1])**2))

The following formulas relate different combinations of $z_0, z_1, \mu_j$ and may prove useful:

In [16]:
A_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wp_z0_mu_j_d.lhs - wp_z1_dj.lhs, 
    wp_z0_mu_j_d.rhs - wp_z1_dj.rhs
)
B_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp_z1_dj.lhs/(wp_z0_mu_j_d.lhs - wp_z1_dj.lhs), 
    wpp_z1_dj.rhs/(wp_z0_mu_j_d.rhs - wp_z1_dj.rhs)
)
C_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    wpp_z0_mu_j_d.lhs/(wp_z0_mu_j_d.lhs - wp_z1_dj.lhs), 
    (wpp_z0_mu_j_d.rhs/(wp_z0_mu_j_d.rhs - wp_z1_dj.rhs)).factor()
)
D_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    B_wpz1_z0_muj_to_d_lam1_gamj.lhs*C_wpz1_z0_muj_to_d_lam1_gamj.lhs, 
    B_wpz1_z0_muj_to_d_lam1_gamj.rhs*C_wpz1_z0_muj_to_d_lam1_gamj.rhs
)
E_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    A_wpz1_z0_muj_to_d_lam1_gamj.lhs/C_wpz1_z0_muj_to_d_lam1_gamj.lhs, 
    A_wpz1_z0_muj_to_d_lam1_gamj.rhs/C_wpz1_z0_muj_to_d_lam1_gamj.rhs
)
F_wpz1_z0_muj_to_d_lam1_gamj = Eq(
    A_wpz1_z0_muj_to_d_lam1_gamj.lhs/B_wpz1_z0_muj_to_d_lam1_gamj.lhs, 
    A_wpz1_z0_muj_to_d_lam1_gamj.rhs/B_wpz1_z0_muj_to_d_lam1_gamj.rhs
)

A_wpz1_z0_muj_to_d_lam1_gamj
B_wpz1_z0_muj_to_d_lam1_gamj
C_wpz1_z0_muj_to_d_lam1_gamj
D_wpz1_z0_muj_to_d_lam1_gamj
E_wpz1_z0_muj_to_d_lam1_gamj
F_wpz1_z0_muj_to_d_lam1_gamj

Eq(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3), -(-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1])))

Eq(\wp'(z1, g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)), -(gamma[j] - lamda[1])*b[2])

Eq(\wp'(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3)), -(b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)/(gamma[j] - lamda[1]))

Eq(\wp'(z1, g2, g3)*\wp'(-z0 + mu[j], g2, g3)/(-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3))**2, (b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)*b[2])

Eq((-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3))**2/\wp'(-z0 + mu[j], g2, g3), (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*(b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)))

Eq((-wp(z1, g2, g3) + wp(-z0 + mu[j], g2, g3))**2/\wp'(z1, g2, g3), (-d[1] - 2*d[2]*lamda[1] - 3*d[3]*lamda[1]**2 - 4*d[4]*lamda[1]**3)/(4*(gamma[j] - lamda[1])**2*b[2]))

Having solved for $\rho(z)$ we now define the function $\rho(z) - \rho(x)$:

In [17]:
rho_z_min_x = Eq(
    rho_z_sqrt_d4.lhs - rho_z_sqrt_d4.lhs.subs(z,x), 
    (rho_z_sqrt_d4.rhs - rho_z_sqrt_d4.rhs.subs(z,x))
)
rho_z_min_x

Eq(-rho(x) + rho(z), \wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*b[2]) - \wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(x - z0, g2, g3))*b[2]))

### Solving for $u,v$

This section also follows the FWM case closely.

From the unified theory, we know that $u(z, \mu_j), v(z, \mu_j)$ solve the following differential equations:

In [18]:
dphiz_unified = Eq(
    Derivative(phi(z, mu[j]), z),
    (Q(-rho(z) + gamma[1], -rho(z) + gamma[N]) - Q(gamma[1] - gamma[j], gamma[N] - gamma[j]))/
    (-2*rho(z) + 2*gamma[j]) - Derivative(Q(-rho(z) + gamma[1], -rho(z) + gamma[N]), gamma[j])
)
dphiz_unified_m4 = Eq(
    Derivative(phi(z, mu[m]), z),
    (
        Q(-rho(z) + gamma[1], -rho(z) + gamma[2], -rho(z) + gamma[3], -rho(z) + gamma[4]) - 
        Q(gamma[1] - gamma[m], gamma[2] - gamma[m], gamma[3] - gamma[m], gamma[4] - gamma[m])
    )/(-rho(z) + gamma[m])/2
    - Derivative(Q(-rho(z) + gamma[1], -rho(z) + gamma[2], -rho(z) + gamma[3], -rho(z) + gamma[4]), gamma[m])
)
duz_unified  = Eq(
    Derivative(u(z, mu[j]), z)/u(z, mu[j]),
    (rhop(z) - rhop(mu[j]))/(2*(rho(z) - rho(mu[j]))) + Derivative(phi(z, mu[j]), z)
)
dvz_unified  = Eq(
    Derivative(v(z, mu[j]), z)/v(z, mu[j]),
    (rhop(z) + rhop(mu[j]))/(2*(rho(z) - rho(mu[j]))) - Derivative(phi(z, mu[j]), z)
)

dphiz_unified
duz_unified
dvz_unified

Eq(Derivative(phi(z, mu[j]), z), (Q(-rho(z) + gamma[1], -rho(z) + gamma[N]) - Q(gamma[1] - gamma[j], gamma[N] - gamma[j]))/(-2*rho(z) + 2*gamma[j]) - Derivative(Q(-rho(z) + gamma[1], -rho(z) + gamma[N]), gamma[j]))

Eq(Derivative(u(z, mu[j]), z)/u(z, mu[j]), (\rho'(z) - \rho'(mu[j]))/(2*rho(z) - 2*rho(mu[j])) + Derivative(phi(z, mu[j]), z))

Eq(Derivative(v(z, mu[j]), z)/v(z, mu[j]), (\rho'(z) + \rho'(mu[j]))/(2*rho(z) - 2*rho(mu[j])) - Derivative(phi(z, mu[j]), z))

We first derive expressions for the logarithmic derivative parts:

In [19]:
dwp_zx_subs = [(diff(wp(z,g2,g3),z), wpp(z,g2,g3)), (diff(wp(x,g2,g3),x), wpp(x,g2,g3))]
log_rho_z_x = Eq(
    log(rho(z) - rho(x)),
    log(-wp(x - z0, g2, g3) + wp(z - z0, g2, g3)) + 
    log(-wp(z1, g2, g3)/b[2]) -
    log(-wp(z1, g2, g3) + wp(z - z0, g2, g3)) - 
    log(-wp(z1, g2, g3) + wp(x - z0, g2, g3))
)

dlog_z_min_x = Eq(
    diff(log_rho_z_x.lhs,z) - diff(log_rho_z_x.lhs,x),
    diff(log_rho_z_x.rhs.subs([(z, z+z0), (x, x+z0)]),z).subs(dwp_zx_subs).subs([(z, z-z0), (x, x-z0)]) - 
    diff(log_rho_z_x.rhs.subs([(z, z+z0), (x, x+z0)]),x).subs(dwp_zx_subs).subs([(z, z-z0), (x, x-z0)])
)
dlog_z_plus_x = Eq(
    diff(log_rho_z_x.lhs,z) + diff(log_rho_z_x.lhs,x),
    diff(log_rho_z_x.rhs.subs([(z, z+z0), (x, x+z0)]),z).subs(dwp_zx_subs).subs([(z, z-z0), (x, x-z0)]) + 
    diff(log_rho_z_x.rhs.subs([(z, z+z0), (x, x+z0)]),x).subs(dwp_zx_subs).subs([(z, z-z0), (x, x-z0)])
)

pw_to_zw_eqs = [
    pw_to_zw_identity_one_sided.subs([(z,t),(x,y)]).subs([(t,z-z0), (y, z1)]),
    pw_to_zw_identity_one_sided.subs([(z,t),(x,y)]).subs([(t,z-z0), (y, x - z0)]),
    pw_to_zw_identity_one_sided.subs([(z,t),(x,y)]).subs([(t,x-z0), (y, z - z0)]),
    pw_to_zw_identity_one_sided.subs([(z,t),(x,y)]).subs([(t,x-z0), (y, z1)])
]

dlog_z_min_x_zeta = Eq(
    (dlog_z_min_x.lhs/2).simplify(),
    ((
        dlog_z_min_x.rhs 
        - pw_to_zw_eqs[0].lhs + pw_to_zw_eqs[0].rhs
        + pw_to_zw_eqs[1].lhs - pw_to_zw_eqs[1].rhs
        - pw_to_zw_eqs[2].lhs + pw_to_zw_eqs[2].rhs
        + pw_to_zw_eqs[3].lhs - pw_to_zw_eqs[3].rhs
    ).collect([wpp(z-z0,g2,g3), wpp(x-z0,g2,g3)], simplify)/2).subs([
        (zeta(x-z,g2,g3), -zeta(z-x,g2,g3))
    ])
)
dlog_z_plus_x_zeta = Eq(
    (dlog_z_plus_x.lhs/2).simplify(),
    ((
        dlog_z_plus_x.rhs 
        - pw_to_zw_eqs[0].lhs + pw_to_zw_eqs[0].rhs
        + pw_to_zw_eqs[1].lhs - pw_to_zw_eqs[1].rhs
        + pw_to_zw_eqs[2].lhs - pw_to_zw_eqs[2].rhs
        - pw_to_zw_eqs[3].lhs + pw_to_zw_eqs[3].rhs
    ).collect([wpp(z-z0,g2,g3), wpp(x-z0,g2,g3)], simplify)/2).subs([
        (zeta(x-z,g2,g3), -zeta(z-x,g2,g3))
    ])
)
    
dlog_z_min_x
print()
dlog_z_plus_x
print()
dlog_z_min_x_zeta
print()
dlog_z_plus_x_zeta

Eq(Derivative(rho(x), x)/(-rho(x) + rho(z)) + Derivative(rho(z), z)/(-rho(x) + rho(z)), \wp'(x - z0, g2, g3)/(-wp(x - z0, g2, g3) + wp(z - z0, g2, g3)) + \wp'(z - z0, g2, g3)/(-wp(x - z0, g2, g3) + wp(z - z0, g2, g3)) - \wp'(z - z0, g2, g3)/(-wp(z1, g2, g3) + wp(z - z0, g2, g3)) + \wp'(x - z0, g2, g3)/(-wp(z1, g2, g3) + wp(x - z0, g2, g3)))

Eq(-Derivative(rho(x), x)/(-rho(x) + rho(z)) + Derivative(rho(z), z)/(-rho(x) + rho(z)), -\wp'(x - z0, g2, g3)/(-wp(x - z0, g2, g3) + wp(z - z0, g2, g3)) + \wp'(z - z0, g2, g3)/(-wp(x - z0, g2, g3) + wp(z - z0, g2, g3)) - \wp'(z - z0, g2, g3)/(-wp(z1, g2, g3) + wp(z - z0, g2, g3)) - \wp'(x - z0, g2, g3)/(-wp(z1, g2, g3) + wp(x - z0, g2, g3)))

Eq((-Derivative(rho(x), x) - Derivative(rho(z), z))/(2*(rho(x) - rho(z))), zeta(-x + z, g2, g3) + zeta(x - z0 - z1, g2, g3)/2 + zeta(x - z0 + z1, g2, g3)/2 - zeta(z - z0 - z1, g2, g3)/2 - zeta(z - z0 + z1, g2, g3)/2)

Eq((Derivative(rho(x), x) - Derivative(rho(z), z))/(2*(rho(x) - rho(z))), zeta(x + z - 2*z0, g2, g3) - zeta(x - z0 - z1, g2, g3)/2 - zeta(x - z0 + z1, g2, g3)/2 - zeta(z - z0 - z1, g2, g3)/2 - zeta(z - z0 + z1, g2, g3)/2)

Then we derive expressions for the gauge function parts:

In [20]:
## -- Difference part --

Q_poly_rho = Q_poly_uv.subs([
    uv_j_rho.subs(j,1).args,
    uv_j_rho.subs(j,2).args
])
Q_poly_rho_gam_m = Q_poly_rho.subs(rho(z), gamma[m])

Q_poly_rho_min_m = Eq(
    ((Q_poly_rho.lhs - Q_poly_rho_gam_m.lhs)/(gamma[m] - rho(z))/2).factor(), 
    - Sum(a[j,k]/4,(j,1,2),(k,1,2))*rho(z)
    - Sum(a[j,k]/4,(j,1,2),(k,1,2))*gamma[m]
    + Sum(a[j]/2,(j,1,2))
    + Sum(a[j,k]*(gamma[j] + gamma[k])/2, (j, 1, 2), (k, 1, 2))/2
)
assert (
    (Q_poly_rho.rhs - Q_poly_rho_gam_m.rhs)/(gamma[m] - rho(z))/2 - Q_poly_rho_min_m.rhs
).doit().simplify() == 0, 'Q_poly_rho_min_m not defined correctly'
Q_poly_rho_min_m = Q_poly_rho_min_m.subs([gammajk_trace_id.args])

## -- Derivative part -- 



dQ_rho_a_p_gamma = Eq(
    Derivative(
        Q(
            -rho(z) + gamma[1], 
            -rho(z) + gamma[2]
        )
        ,gamma[m]
    ),
    a[m] + Sum(gamma[k]*a[m, k], (k, 1, 2)) - rho(z)*Sum(a[m, k], (k, 1, 2))
)
# Q_poly_gam_rho = Q_poly_uv.subs([(u(z,mu[_j])*v(z,mu[_j]), gamma[_j] - rho(z)) for _j in range(1,5)])

dQ_checks = [
    (
        diff(Q_poly_rho.rhs.doit(), gamma[_m]).subs(ajk_syms) - 
        dQ_rho_a_p_gamma.rhs.subs(m, _m).doit().subs(ajk_syms)
    ).simplify() != 0
    for _m in range(1, N_modes + 1)
]
if any(dQ_checks):
    raise ValueError('dQ_checks failed')
    
    
## -- Phi derivative --

dphi_m_Q = dphiz_unified.subs([(N,2),(j,m)]).subs([
    Q_poly_rho_min_m.args, 
    dQ_rho_a_p_gamma.args
])
dphi_m_Q = Eq(
    dphi_m_Q.lhs, 
    sum(_.factor() for _ in dphi_m_Q.rhs.collect(rho(z)).args)
    .subs([
        Q_poly_rho_min_m.args, 
        dQ_rho_a_p_gamma.args
    ])
    .collect(rho(z))
)
Lams = [
    Eq(Lamda[_j, m], dphi_m_Q.rhs.coeff(rho(z),_j) ) 
    for _j in range(2)
]
dphi_m_Q_Lam = dphi_m_Q.subs([(_eq.rhs, _eq.lhs) for _eq in Lams])


Q_poly_rho
Q_poly_rho_gam_m
Q_poly_rho_min_m

dQ_rho_a_p_gamma


dphi_m_Q
for _eq in Lams:
    _eq
dphi_m_Q_Lam


Eq(Q(-rho(z) + gamma[1], -rho(z) + gamma[2]), a[0] + Sum((-rho(z) + gamma[j])*a[j], (j, 1, 2)) + Sum((-rho(z) + gamma[j])*(-rho(z) + gamma[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(Q(gamma[1] - gamma[m], gamma[2] - gamma[m]), a[0] + Sum((gamma[j] - gamma[m])*a[j], (j, 1, 2)) + Sum((gamma[j] - gamma[m])*(gamma[k] - gamma[m])*a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq((Q(-rho(z) + gamma[1], -rho(z) + gamma[2]) - Q(gamma[1] - gamma[m], gamma[2] - gamma[m]))/(2*(-rho(z) + gamma[m])), -rho(z)*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)) - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)) + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 + Sum(a[j]/2, (j, 1, 2)))

Eq(Derivative(Q(-rho(z) + gamma[1], -rho(z) + gamma[2]), gamma[m]), -rho(z)*Sum(a[m, k], (k, 1, 2)) + a[m] + Sum(a[m, k]*gamma[k], (k, 1, 2)))

Eq(Derivative(phi(z, mu[m]), z), (Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))*rho(z) - a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)) + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2)))

Eq(Lamda[0, m], -a[m] - gamma[m]*Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)) + Sum(a[j, j]*gamma[j], (j, 1, 2))/2 - Sum(a[m, k]*gamma[k], (k, 1, 2)) + Sum(a[j]/2, (j, 1, 2)))

Eq(Lamda[1, m], Sum(a[m, k], (k, 1, 2)) - Sum(a[j, k]/4, (j, 1, 2), (k, 1, 2)))

Eq(Derivative(phi(z, mu[m]), z), rho(z)*Lamda[1, m] + Lamda[0, m])

The $\Lambda_{j,m}$ are the same as those defined for FWM but with 2 modes not 4. In the case of $\Lambda_{0,m}$ there is one term that looks at first glance to be different but when one consider $\gamma_1+\gamma_2=0$ and $a_{2,1}=a{1,2}$ it can be confirmed to be identical in form to the FWM case.

The $\Lambda_{j,m}$ terms can be summed over $m$ to give the results below. Unlike the FWM case, the sum over $\Lambda_{1,m}$ is not $0$ which is particularly notable for gauge transforms:

In [21]:
# Lambda_0 sum
Lam0_sum_original = Eq(
    Sum(Lams[0].lhs, (m, 1, N_modes)), 
    Sum(Lams[0].rhs, (m, 1, N_modes))
)
Lam0_sum = Eq(
    Sum(Lamda[0, m], (m, 1, N_modes)), 
    - Sum(gamma[j], (j, 1, N_modes))*Sum(a[j, k]/2, (j, 1, N_modes), (k, 1, N_modes))/2
    - Sum(a[j, k]*gamma[k], (j, 1, N_modes), (k, 1, N_modes))
    + Sum((gamma[j]+gamma[k])*a[j, k]/2, (j, 1, N_modes), (k, 1, N_modes)) 
).subs([gammajk_trace_id.args])
assert (Lam0_sum.rhs - Lam0_sum_original.rhs).doit().subs(ajk_syms).expand() == 0, 'Lam0_sum check failed'

Lam0_sum_b = Eq(
    Lam0_sum.lhs, 
    Lam0_sum.rhs.subs([sum_gamma_j_0.args])
)
Lam0_sum_c = Eq(
    Lam0_sum_b.lhs, 
    Lam0_sum_b.rhs.doit().subs(ajk_syms).factor().subs([sum_gamma_j_0.doit().args])
)

# Lambda_1 sum
Lam1_sum = Eq(
    Sum(Lams[1].lhs, (m, 1, N_modes)), 
    Sum(Lams[1].rhs, (m, 1, N_modes)).doit() - (b_j_coeffs[2].rhs.doit() - b_j_coeffs[2].rhs)
)
Lam1_sum_b = Lam1_sum.subs(b_j_coeffs[2].rhs, b_j_coeffs[2].lhs)


Lam0_sum
Lam0_sum_b
Lam0_sum_c

Lam1_sum
Lam1_sum_b

Eq(Sum(Lamda[0, m], (m, 1, 2)), Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(gamma[j], (j, 1, 2))*Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2))/2 - Sum(a[j, k]*gamma[k], (j, 1, 2), (k, 1, 2)))

Eq(Sum(Lamda[0, m], (m, 1, 2)), Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j, k]*gamma[k], (j, 1, 2), (k, 1, 2)))

Eq(Sum(Lamda[0, m], (m, 1, 2)), 0)

Eq(Sum(Lamda[1, m], (m, 1, 2)), Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(Sum(Lamda[1, m], (m, 1, 2)), b[2])

We now put these parts together to solve for $u,v$:

In [22]:
duz_unified_rho_only = duz_unified.subs(*dphi_m_Q_Lam.subs(m,j).args)
dvz_unified_rho_only = dvz_unified.subs(*dphi_m_Q_Lam.subs(m,j).args)

## -- Build zeta expansion for log diffs of u, v --

duz_unified_zeta = duz_unified.expand().subs([
    dlog_z_plus_x_zeta.subs([
        (diff(rho(z),z),rhop(z)), (diff(rho(x),x),rhop(x))
    ]).subs(x, mu[j]).simplify().expand().args,
    dphi_m_Q_Lam.subs([(m,j), rho_zeta.args]).args
])
dvz_unified_zeta = dvz_unified.expand().subs([
    dlog_z_min_x_zeta.subs([
        (diff(rho(z),z),rhop(z)), (diff(rho(x),x),rhop(x))
    ]).subs(x, mu[j]).simplify().expand().args,
    dphi_m_Q_Lam.subs([(m,j), rho_zeta.args]).args
])
duz_unified_zeta = Eq(
    duz_unified_zeta.lhs, 
    duz_unified_zeta.rhs.expand().collect([Lamda[1,j]/b[2]]).subs(zeta(-z+z1+z0,g2,g3), -zeta(z-z1-z0,g2,g3))
)
dvz_unified_zeta = Eq(
    dvz_unified_zeta.lhs, 
    dvz_unified_zeta.rhs.expand().collect([Lamda[1,j]/b[2]]).subs(zeta(-z+z1+z0,g2,g3), -zeta(z-z1-z0,g2,g3))
)


## -- Convert zetas to log diff sigmas --

zw_dlog_subs = [
    zw_dlog_sigma_identity.subs([(x, z1)]).args,
    zw_dlog_sigma_identity.subs([(x, mu[j])]).args,
    zw_dlog_sigma_identity.subs([(x, z0 - z1)]).args,
    zw_dlog_sigma_identity.subs([(x, z0 + z1)]).args,
    zw_dlog_sigma_identity.subs([(x, 2*z0 - mu[j])]).args
]

duz_unified_zeta_b = duz_unified_zeta.subs(zw_dlog_subs)
dvz_unified_zeta_b = dvz_unified_zeta.subs(zw_dlog_subs)


## -- Combine log diff sigmas --

dlog_sub_d4 = Eq(
    Derivative(log(sigma(z - z0 + z1, g2, g3)), z)*Lamda[1, j]/b[2]
    -Derivative(log(sigma(z - z1 - z0, g2, g3)), z)*Lamda[1, j]/b[2],
    Derivative(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3)), z)*Lamda[1, j]/b[2]
)
dlog_sub_no_d4_1 = Eq(
    Derivative(log(sigma(z - 2*z0 + mu[j], g2, g3)), z)
    - Derivative(log(sigma(z - z1 - z0, g2, g3)), z)/2
    - Derivative(log(sigma(z - z0 + z1, g2, g3)), z)/2,
    Derivative(log(
        sigma(z - 2*z0 + mu[j], g2, g3)/
        sqrt(sigma(z - z1 - z0, g2, g3)*sigma(z - z0 + z1, g2, g3))
    ), z)
)
dlog_sub_no_d4_2 =Eq(
    Derivative(log(sigma(z - mu[j], g2, g3)), z)
    - Derivative(log(sigma(z - z1 - z0, g2, g3)), z)/2
    - Derivative(log(sigma(z - z0 + z1, g2, g3)), z)/2,
    Derivative(log(
        sigma(z - mu[j], g2, g3)/
        sqrt(sigma(z - z1 -z0, g2, g3)*sigma(z - z0 + z1, g2, g3))
    ), z)
)
_log_combined_subs = [dlog_sub_d4.args, dlog_sub_no_d4_1.args, dlog_sub_no_d4_2.args]

duz_unified_zeta_c = duz_unified_zeta_b.expand().subs(_log_combined_subs)
dvz_unified_zeta_c = dvz_unified_zeta_b.expand().subs(_log_combined_subs)


## -- Substitute short param names for non-log parts --

_duz_ts = duz_unified_zeta_b.expand().rhs.args
duz_unified_zeta_b_non_log_terms = sum(_t for _t in _duz_ts if 'log' not in str(_t))
r0j_non_log_part = Eq(r[0,j], duz_unified_zeta_b_non_log_terms)
r1j_log_part = Eq(r[1,j], Lamda[1,j]/b[2])
r2j_log_part = Eq(r[2,j], Lamda[1,j]/b[2] - Rational(1,2))

duz_unified_zeta_c = Eq(
    duz_unified_zeta_c.lhs, 
    (duz_unified_zeta_c.rhs - r0j_non_log_part.rhs + r0j_non_log_part.lhs)
    .subs(b[2], solve(r1j_log_part, b[2])[0])
)
dvz_unified_zeta_c = Eq(
    dvz_unified_zeta_c.lhs, 
    (dvz_unified_zeta_c.rhs + r0j_non_log_part.rhs - r0j_non_log_part.lhs)
    .subs(b[2], solve(r1j_log_part, b[2])[0])
)


## -- Define the solutions after integrating --

uz_sol = Eq(
    u(z, mu[j]),
    sigma(z - 2*z0 + mu[j], g2, g3)/sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3))*
    exp(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + r[0, j]*z + r[3,j]/2 + epsilon[j])
)
vz_sol = Eq(
    v(z, mu[j]),
    sigma(z - mu[j], g2, g3)/sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3))*
    exp(-log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - r[0, j]*z + r[3,j]/2 - epsilon[j])
)

uz_sol_no_sqrt = Eq(
    u(z, mu[j]),
    sigma(z - 2*z0 + mu[j], g2, g3)/sigma(z - z0 - z1, g2, g3)*
    exp(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[2, j] + r[0, j]*z + r[3,j]/2 + epsilon[j])
)
vz_sol_no_sqrt = Eq(
    v(z, mu[j]),
    sigma(z - mu[j], g2, g3)/sigma(z - z0 + z1, g2, g3)*
    exp(-log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[2, j] - r[0, j]*z + r[3,j]/2 - epsilon[j])
)

uz_sol_frac_exp = Eq(
    u(z, mu[j]),
    sigma(z - 2*z0 + mu[j], g2, g3)*
    sigma(z - z0 + z1, g2, g3)**r[2,j]*
    sigma(z - z0 - z1, g2, g3)**(-r[2,j]-1)*
    exp(r[0, j]*z + r[3,j]/2 + epsilon[j])
)
vz_sol_frac_exp = Eq(
    v(z, mu[j]),
    sigma(z - mu[j], g2, g3)*
    sigma(z - z0 - z1, g2, g3)**r[2,j]*
    sigma(z - z0 + z1, g2, g3)**(-r[2,j]-1)*
    exp( - r[0, j]*z + r[3,j]/2 - epsilon[j])
)

## -- Check they solve the equation we started from --

zw_dlog_sigma_id_b = Eq(zw_dlog_sigma_identity.rhs, zw_dlog_sigma_identity.lhs)
sig_zw_subs = [zw_dlog_sigma_id_b.subs(x,_x).doit().args for _x in [z0 - z1, z0 + z1, 2*z0 - mu[j], mu[j]]]
r0r1r2_subs = [r0j_non_log_part.args, r1j_log_part.args, r2j_log_part.args]

u_sol_check = (
    duz_unified_zeta.lhs.subs([uz_sol.args]).doit().simplify().subs(sig_zw_subs).subs(r0r1r2_subs)
    - duz_unified_zeta.rhs
).simplify() == 0
v_sol_check = (
    dvz_unified_zeta.lhs.subs([vz_sol.args]).doit().simplify().subs(sig_zw_subs).subs(r0r1r2_subs)
    - dvz_unified_zeta.rhs
).simplify() == 0
if not u_sol_check:
    raise ValueError('u_sol_check failed')
if not v_sol_check:
    raise ValueError('v_sol_check failed')
    
u_sol_no_sqrt_check = (
    duz_unified_zeta.lhs.subs([uz_sol_no_sqrt.args]).doit().simplify().subs(sig_zw_subs).subs(r0r1r2_subs)
    - duz_unified_zeta.rhs
).simplify() == 0
v_sol_no_sqrt_check = (
    dvz_unified_zeta.lhs.subs([vz_sol_no_sqrt.args]).doit().simplify().subs(sig_zw_subs).subs(r0r1r2_subs)
    - dvz_unified_zeta.rhs
).simplify() == 0
if not u_sol_no_sqrt_check:
    raise ValueError('u_sol_no_sqrt_check failed')
if not v_sol_no_sqrt_check:
    raise ValueError('v_sol_no_sqrt_check failed')
    
u_sol_frac_exp_check = (
    (duz_unified_zeta.lhs.subs([uz_sol_frac_exp.args]).doit()
    .simplify().subs(r0r1r2_subs).expand().subs(sig_zw_subs)
    ).simplify().subs(sig_zw_subs)
    - duz_unified_zeta.rhs
).simplify() == 0
v_sol_frac_exp_check = (
    (dvz_unified_zeta.lhs.subs([vz_sol_frac_exp.args]).doit()
    .simplify().subs(r0r1r2_subs).expand().subs(sig_zw_subs)
    ).simplify().subs(sig_zw_subs)
    - dvz_unified_zeta.rhs
).simplify() == 0
if not u_sol_frac_exp_check:
    raise ValueError('u_sol_frac_exp_check failed')
if not v_sol_frac_exp_check:
    raise ValueError('v_sol_frac_exp_check failed')

duz_unified_rho_only
dvz_unified_rho_only

duz_unified_zeta
print()
dvz_unified_zeta
print()

duz_unified_zeta_b
print()
dvz_unified_zeta_b
print()

r0j_non_log_part
r1j_log_part
r2j_log_part
print()

duz_unified_zeta_c
print()
dvz_unified_zeta_c
print()

uz_sol
vz_sol
print()
uz_sol_no_sqrt
vz_sol_no_sqrt
print()
uz_sol_frac_exp
vz_sol_frac_exp

Eq(Derivative(u(z, mu[j]), z)/u(z, mu[j]), (\rho'(z) - \rho'(mu[j]))/(2*rho(z) - 2*rho(mu[j])) + rho(z)*Lamda[1, j] + Lamda[0, j])

Eq(Derivative(v(z, mu[j]), z)/v(z, mu[j]), (\rho'(z) + \rho'(mu[j]))/(2*rho(z) - 2*rho(mu[j])) - rho(z)*Lamda[1, j] - Lamda[0, j])

Eq(Derivative(u(z, mu[j]), z)/u(z, mu[j]), (-2*zeta(z1, g2, g3) - zeta(z - z0 - z1, g2, g3) + zeta(z - z0 + z1, g2, g3))*Lamda[1, j]/b[2] + zeta(z - 2*z0 + mu[j], g2, g3) - zeta(z - z0 - z1, g2, g3)/2 - zeta(z - z0 + z1, g2, g3)/2 - zeta(-z0 - z1 + mu[j], g2, g3)/2 - zeta(-z0 + z1 + mu[j], g2, g3)/2 + Lamda[0, j] + Lamda[1, j]*lamda[1])

Eq(Derivative(v(z, mu[j]), z)/v(z, mu[j]), (2*zeta(z1, g2, g3) + zeta(z - z0 - z1, g2, g3) - zeta(z - z0 + z1, g2, g3))*Lamda[1, j]/b[2] + zeta(z - mu[j], g2, g3) - zeta(z - z0 - z1, g2, g3)/2 - zeta(z - z0 + z1, g2, g3)/2 + zeta(-z0 - z1 + mu[j], g2, g3)/2 + zeta(-z0 + z1 + mu[j], g2, g3)/2 - Lamda[0, j] - Lamda[1, j]*lamda[1])

Eq(Derivative(u(z, mu[j]), z)/u(z, mu[j]), (-2*zeta(z1, g2, g3) - Derivative(log(sigma(z - z0 - z1, g2, g3)), z) + Derivative(log(sigma(z - z0 + z1, g2, g3)), z))*Lamda[1, j]/b[2] - zeta(-z0 - z1 + mu[j], g2, g3)/2 - zeta(-z0 + z1 + mu[j], g2, g3)/2 + Derivative(log(sigma(z - 2*z0 + mu[j], g2, g3)), z) - Derivative(log(sigma(z - z0 - z1, g2, g3)), z)/2 - Derivative(log(sigma(z - z0 + z1, g2, g3)), z)/2 + Lamda[0, j] + Lamda[1, j]*lamda[1])

...

Eq(v(z, mu[j]), sigma(z - mu[j], g2, g3)*exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j] + r[3, j]/2)/sqrt(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)))

Eq(u(z, mu[j]), sigma(z - 2*z0 + mu[j], g2, g3)*exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[2, j] + epsilon[j] + r[3, j]/2)/sigma(z - z0 - z1, g2, g3))

Eq(v(z, mu[j]), sigma(z - mu[j], g2, g3)*exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[2, j] - epsilon[j] + r[3, j]/2)/sigma(z - z0 + z1, g2, g3))

Eq(u(z, mu[j]), sigma(z - 2*z0 + mu[j], g2, g3)*sigma(z - z0 - z1, g2, g3)**(-r[2, j] - 1)*sigma(z - z0 + z1, g2, g3)**r[2, j]*exp(z*r[0, j] + epsilon[j] + r[3, j]/2))

Eq(v(z, mu[j]), sigma(z - mu[j], g2, g3)*sigma(z - z0 - z1, g2, g3)**r[2, j]*sigma(z - z0 + z1, g2, g3)**(-r[2, j] - 1)*exp(-z*r[0, j] - epsilon[j] + r[3, j]/2))

From these, we can derive expressions for the product of $u,v$:

In [23]:
dlog_uv = Eq(
    duz_unified_zeta_b.lhs + dvz_unified_zeta_b.lhs, 
    (duz_unified_zeta_b.rhs + dvz_unified_zeta_b.rhs).expand()
)
dlog_uv_b = Eq(
    Derivative(log(u(z, mu[j])*v(z, mu[j])), z),
    Derivative(log(
        sigma(z - 2*z0 + mu[j], g2, g3)*sigma(z - mu[j], g2, g3)/
        (sigma(z - z1 - z0, g2, g3)*sigma(z - z0 + z1, g2, g3))
    ), z)
)

dlog_uv_check_lhs = (dlog_uv.lhs - dlog_uv_b.lhs).doit().simplify() == 0
dlog_uv_check_rhs = (dlog_uv.rhs - dlog_uv_b.rhs).doit().simplify() == 0
dlog_uv_check = dlog_uv_check_lhs and dlog_uv_check_rhs
if not dlog_uv_check:
    raise ValueError('dlog_uv_check failed')
    
uv_sol = Eq(
    u(z, mu[j])*v(z, mu[j]),
    sigma(z - mu[j], g2, g3)*sigma(z - 2*z0 + mu[j], g2, g3)/
    (sigma(z - z1 - z0, g2, g3)*sigma(z - z0 + z1, g2, g3))*exp(r[3,j])
)

dlog_uv
dlog_uv_b
uv_sol

Eq(Derivative(v(z, mu[j]), z)/v(z, mu[j]) + Derivative(u(z, mu[j]), z)/u(z, mu[j]), Derivative(log(sigma(z - mu[j], g2, g3)), z) + Derivative(log(sigma(z - 2*z0 + mu[j], g2, g3)), z) - Derivative(log(sigma(z - z0 - z1, g2, g3)), z) - Derivative(log(sigma(z - z0 + z1, g2, g3)), z))

Eq(Derivative(log(u(z, mu[j])*v(z, mu[j])), z), Derivative(log(sigma(z - mu[j], g2, g3)*sigma(z - 2*z0 + mu[j], g2, g3)/(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3))), z))

Eq(u(z, mu[j])*v(z, mu[j]), sigma(z - mu[j], g2, g3)*sigma(z - 2*z0 + mu[j], g2, g3)*exp(r[3, j])/(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)))

Which, when compared to the original expression for $u(z, \mu_j)v(z, \mu_j)$, fixes the constant $r_{2,j}$:

In [24]:
# -- Convert sigma sol to wp form --

sig_pw_z0_mu_j_z1 = Eq(
    sigma_p_identity.rhs.subs([(x,z-z0),(y, mu[j] - z0)])/
    sigma_p_identity.rhs.subs([(x,z-z0),(y, z1)]),
    sigma_p_identity.lhs.subs([(x,z-z0),(y, mu[j] - z0)])/
    sigma_p_identity.lhs.subs([(x,z-z0),(y, z1)])
)
uv_sol_wp = Eq(uv_sol.lhs, uv_sol.rhs * sig_pw_z0_mu_j_z1.rhs/sig_pw_z0_mu_j_z1.lhs)


## -- Compare with original form that used mu_j and solve for r2j --

r3j_conistency = Eq(
    uv_mu_j_z1p.lhs.simplify()/uv_sol_wp.lhs, 
    uv_mu_j_z1p.rhs.simplify()/uv_sol_wp.rhs
)
exp_r_3_j = Eq(exp(r[3,j]), solve(r3j_conistency, exp(r[3,j]))[0])
exp_r_3_j_sigma = exp_r_3_j.subs([
    (
        -sigma_p_identity.subs([(x,z1),(y, mu[j] - z0)]).lhs,
        -sigma_p_identity.subs([(x,z1),(y, mu[j] - z0)]).rhs
    ),
    pwp_sigma_dbl_ratio.subs(z,z1).args
])
exp_r_3_j_over_2 = Eq(
    exp(r[3,j]/2), 
    sqrt(wpp(z1, g2, g3))/sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))/sqrt(b[2]) *
    sigma(z1, g2, g3) / sigma(-z0 + mu[j], g2, g3)
)
uv_mu_j_z0 = uv_mu_j_z1p.subs([(z,z0), (1/(-wp(0, g2, g3) + wp(z1, g2, g3)),0)])
uv_mu_j_z_at_0 = uv_uv_0_j.subs([(z, mu[j]),(u(mu[j], mu[j])*v(mu[j], mu[j]),0)])

exp_r_3_j
exp_r_3_j_sigma
exp_r_3_j_over_2
uv_mu_j_z0
uv_mu_j_z_at_0

Eq(exp(r[3, j]), \wp'(z1, g2, g3)*sigma(z1, g2, g3)**2/((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(-z0 + mu[j], g2, g3)**2*b[2]))

Eq(exp(r[3, j]), sigma(2*z1, g2, g3)/(sigma(-z0 + z1 + mu[j], g2, g3)*sigma(z0 + z1 - mu[j], g2, g3)*b[2]))

Eq(exp(r[3, j]/2), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(-z0 + mu[j], g2, g3)*sqrt(b[2])))

Eq(u(z0, mu[j])*v(z0, mu[j]), \wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*b[2]))

Eq(0, u(0, mu[j])*v(0, mu[j]) - \wp'(z1, g2, g3)/((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*b[2]) + \wp'(z1, g2, g3)/((-wp(z0, g2, g3) + wp(z1, g2, g3))*b[2]))

and this constant leads to these formulas for $u,v$:

In [25]:
uz_sol_no_sqrt_b = uz_sol_no_sqrt.expand().subs([exp_r_3_j_over_2.args]).simplify()
vz_sol_no_sqrt_b = vz_sol_no_sqrt.expand().subs([exp_r_3_j_over_2.args]).simplify()

u_sqrt_wp_z0_z1 = Eq(
    u(z, mu[j]),
    1/sqrt(b[2])*
    sqrt(wpp(z1, g2, g3))/
    sqrt((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3)))/
    sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
    *sigma(z - 2*z0 + mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + epsilon[j])
)
v_sqrt_wp_z0_z1 = Eq(
    v(z, mu[j]),
    1/sqrt(b[2])*
    sqrt(wpp(z1, g2, g3))/
    sqrt((wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3)))/
    sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))
    *sigma(z - mu[j], g2, g3)/(sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3))
    *exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j])
)

assert (((uz_sol_no_sqrt_b.rhs/u_sqrt_wp_z0_z1.rhs).simplify()).subs([
    r2j_log_part.subs(r1j_log_part.rhs, r1j_log_part.lhs).args,
    sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
])**2).simplify() == 1, 'uj expressions are different'
assert (((vz_sol_no_sqrt_b.rhs/v_sqrt_wp_z0_z1.rhs).simplify()).subs([
    r2j_log_part.subs(r1j_log_part.rhs, r1j_log_part.lhs).args,
    sigma_p_identity.subs([(y,z1), (x, z-z0)]).args
])**2).simplify() == 1, 'vj expressions are different'

uz_sol_no_sqrt_b
vz_sol_no_sqrt_b
u_sqrt_wp_z0_z1
v_sqrt_wp_z0_z1

Eq(u(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - 2*z0 + mu[j], g2, g3)*exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[2, j] + epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(-z0 + mu[j], g2, g3)*sigma(z - z0 - z1, g2, g3)*sqrt(b[2])))

Eq(v(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z1, g2, g3)*sigma(z - mu[j], g2, g3)*exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[2, j] - epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(-z0 + mu[j], g2, g3)*sigma(z - z0 + z1, g2, g3)*sqrt(b[2])))

Eq(u(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[j], g2, g3)*exp(z*r[0, j] + log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*sqrt(b[2])))

Eq(v(z, mu[j]), sqrt(\wp'(z1, g2, g3))*sigma(z - mu[j], g2, g3)*exp(-z*r[0, j] - log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*sqrt(b[2])))

In [26]:
uv_wp_z0z1muj = Eq(uz_sol_no_sqrt_b.lhs*vz_sol_no_sqrt_b.lhs,
    (uz_sol_no_sqrt_b.rhs*vz_sol_no_sqrt_b.rhs).simplify()*(
        sigma_p_identity.subs([(x,z1),(y, z - z0)]).rhs/
        sigma_p_identity.subs([(x,z1),(y, z - z0)]).lhs
    ).subs([
        (sigma(-z+z0+z1,g2,g3), -sigma(z-z0-z1,g2,g3))
    ])*(
        sigma_p_identity.subs([(x,mu[j]-z0),(y, z - z0)]).lhs/
        sigma_p_identity.subs([(x,mu[j]-z0),(y, z - z0)]).rhs
    ).subs([
        (sigma(-z+mu[j],g2,g3), -sigma(z-mu[j],g2,g3))
    ])
)

uv0_check = uv_mu_j_z_at_0.subs(*uv_wp_z0z1muj.subs(z,0).args).subs(wp(-z0,g2,g3), wp(z0,g2,g3)).simplify()
if not uv0_check:
    raise ValueError('uv0_check failed')

uv_wp_z0z1muj


Eq(u(z, mu[j])*v(z, mu[j]), (wp(z - z0, g2, g3) - wp(-z0 + mu[j], g2, g3))*\wp'(z1, g2, g3)/((-wp(z1, g2, g3) + wp(z - z0, g2, g3))*(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*b[2]))

In [27]:
uv_wp_mu1_mu2_const = Eq(
    uv_wp_z0z1muj.lhs.subs(j,1) - uv_wp_z0z1muj.lhs.subs(j,2),
    (uv_wp_z0z1muj.rhs.subs(j,1) - uv_wp_z0z1muj.rhs.subs(j,2)).simplify().factor()
)

uv_wp_mu1_mu2_const

Eq(u(z, mu[1])*v(z, mu[1]) - u(z, mu[2])*v(z, mu[2]), (wp(-z0 + mu[1], g2, g3) - wp(-z0 + mu[2], g2, g3))*\wp'(z1, g2, g3)/((-wp(z1, g2, g3) + wp(-z0 + mu[1], g2, g3))*(-wp(z1, g2, g3) + wp(-z0 + mu[2], g2, g3))*b[2]))

We can derive an expression for the ratio $\frac{u}{v}$:

In [28]:
dlog_u_over_v = Eq(
    duz_unified_zeta_b.lhs - dvz_unified_zeta_b.lhs,
    (duz_unified_zeta_b.rhs - dvz_unified_zeta_b.rhs).collect(Lamda[1,j]/b[2], factor)
)
combined_dlog = Eq(
    - Derivative(log(sigma(z - z1 - z0, g2, g3)), z) + Derivative(log(sigma(z - z0 + z1, g2, g3)), z),
    Derivative(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3)), z)
)
dlog_u_over_v_id = Eq(
    diff(log(u(z,mu[j])/v(z,mu[j])),z).doit().expand(),
    Derivative(log(u(z,mu[j])/v(z,mu[j])),z)
)

dlog_u_over_v_b = dlog_u_over_v.subs([
    combined_dlog.args, 
    combined_dlog.subs(z1,mu[j]).args,
    dlog_u_over_v_id.args,
]).expand()

dlog_u_over_v_b = Eq(
    dlog_u_over_v_b.lhs, 
    (dlog_u_over_v_b.rhs - 2*r0j_non_log_part.rhs + 2*r0j_non_log_part.lhs)
    .subs(b[2], solve(r1j_log_part, b[2])[0])
)

log_u_over_v_j = Eq(
    log(u(z, mu[j])/v(z, mu[j])),
    log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j]*2 + 
    log(sigma(z - 2*z0 + mu[j], g2, g3)/sigma(z - mu[j], g2, g3)) + 
    r[0, j]*2*z + r[3,j]
)
u_over_v_j = Eq(
    u(z, mu[j])/v(z, mu[j]),
    sigma(z - 2*z0 + mu[j], g2, g3)/sigma(z - mu[j], g2, g3)*
    exp(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*2*r[1, j] + 2*r[0, j]*z + 2*epsilon[j])
)

u_over_v_check = (dlog_u_over_v_b.lhs.subs(*u_over_v_j.args) - dlog_u_over_v_b.rhs).doit().simplify() == 0
if not u_over_v_check:
    u_over_v_check
    raise ValueError('u_over_v_check failed')
    
u_over_v_check_2 = dlog_u_over_v_b.subs(*u_over_v_j.args).doit().simplify()
if not u_over_v_check_2:
    u_over_v_check
    raise ValueError('u_over_v_check_2 failed')


# dlog_u_over_v
print()
# r0j_non_log_part
# r1j_log_part
dlog_u_over_v_b
# log_u_over_v_j
u_over_v_j

Eq(Derivative(log(u(z, mu[j])/v(z, mu[j])), z), 2*Derivative(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3)), z)*r[1, j] - Derivative(log(sigma(z - mu[j], g2, g3)), z) + Derivative(log(sigma(z - 2*z0 + mu[j], g2, g3)), z) + 2*r[0, j])

Eq(u(z, mu[j])/v(z, mu[j]), sigma(z - 2*z0 + mu[j], g2, g3)*exp(2*z*r[0, j] + 2*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + 2*epsilon[j])/sigma(z - mu[j], g2, g3))

and from the product and ratio formulas we get the expressions for the squares:

In [29]:
u_sqrd = Eq(uv_sol.lhs*u_over_v_j.lhs, (uv_sol.rhs*u_over_v_j.rhs).simplify())
v_sqrd = Eq(uv_sol.lhs/u_over_v_j.lhs, (uv_sol.rhs/u_over_v_j.rhs).simplify())

u_sqrd_check = Eq(u_sqrd.lhs/uz_sol.lhs**2, (u_sqrd.rhs/uz_sol.rhs**2).simplify())
v_sqrd_check = Eq(v_sqrd.lhs/vz_sol.lhs**2, (v_sqrd.rhs/vz_sol.rhs**2).simplify())

if not u_sqrd_check:
    raise ValueError('u_sqrd_check failed')
if not v_sqrd_check:
    raise ValueError('v_sqrd_check failed')

u_sqrd
v_sqrd

Eq(u(z, mu[j])**2, sigma(z - 2*z0 + mu[j], g2, g3)**2*exp(2*z*r[0, j] + 2*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] + 2*epsilon[j] + r[3, j])/(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)))

Eq(v(z, mu[j])**2, sigma(z - mu[j], g2, g3)**2*exp(-2*z*r[0, j] - 2*log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, j] - 2*epsilon[j] + r[3, j])/(sigma(z - z0 - z1, g2, g3)*sigma(z - z0 + z1, g2, g3)))

### The sum over $r_{1,j}$

In [30]:
sum_r1j = Eq(
    Sum(r1j_log_part.lhs,(j,1,2)), 
    Sum(r1j_log_part.rhs,(j,1,2))
)

sum_r1j_1 = Eq(
    sum_r1j.lhs, 
    sum_r1j.rhs
    .doit().simplify()
    .subs([Lam1_sum_b.doit().args])
)

r1j_log_part
sum_r1j
Lam1_sum_b
sum_r1j_1

Eq(r[1, j], Lamda[1, j]/b[2])

Eq(Sum(r[1, j], (j, 1, 2)), Sum(Lamda[1, j]/b[2], (j, 1, 2)))

Eq(Sum(Lamda[1, m], (m, 1, 2)), b[2])

Eq(Sum(r[1, j], (j, 1, 2)), 1)

In [31]:
Lam0_sum
Lam0_sum_b
Lam0_sum_c

Lam1_sum
Lam1_sum_b

Eq(Sum(Lamda[0, m], (m, 1, 2)), Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(gamma[j], (j, 1, 2))*Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2))/2 - Sum(a[j, k]*gamma[k], (j, 1, 2), (k, 1, 2)))

Eq(Sum(Lamda[0, m], (m, 1, 2)), Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j, k]*gamma[k], (j, 1, 2), (k, 1, 2)))

Eq(Sum(Lamda[0, m], (m, 1, 2)), 0)

Eq(Sum(Lamda[1, m], (m, 1, 2)), Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(Sum(Lamda[1, m], (m, 1, 2)), b[2])

## The case where the quartic $\rho$ term cancels

There will be no $\rho(z)^4$ term in the differential equation when $d_4=0$. This case is simpler to solve.
For $d_4=0$ we require $b_2 = 0$, which is a requirement of the parameters, i.e., it cannot be achieved in general through tailored initial conditions of the modes. However, unlike the FWM case which leaves a cubic when $d_4 = 0$, in this case if $b_2 = 0$ then this also forces $d_3 = 0$ and this levaes a quadratic not a cubic, thus the solutions are not elliptic functions. We wont explore this further for the time being. 

In [32]:
for _eq in b_j_coeffs:
    _eq

for _eq in c_j_coeffs:
    _eq

for _eq in d_j_coeffs:
    _eq

Eq(b[0], (gamma[1] - gamma[2])**2*(Sum(a[j, j]/4, (j, 1, 2)) - Sum(a[j, k]/8, (j, 1, 2), (k, 1, 2))) + a[0] + Sum(a[j]*gamma[j], (j, 1, 2)))

Eq(b[1], -Sum(a[j, j]*gamma[j], (j, 1, 2)) - Sum(a[j], (j, 1, 2)))

Eq(b[2], Sum(a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(c[0], Product(gamma[j], (j, 1, 2)))

Eq(c[1], 0)

...

Eq(d[0], b[0]**2 - 4*c[0])

Eq(d[1], 2*b[0]*b[1] - 4*c[1])

Eq(d[2], 2*b[0]*b[2] + b[1]**2 - 4*c[2])

Eq(d[3], 2*b[1]*b[2])

Eq(d[4], b[2]**2)

In [33]:
for _eq in d_j_coeffs:
    _eq.subs([_.args for _ in c_j_coeffs]).subs(b[2],0)

Eq(d[0], b[0]**2 - 4*Product(gamma[j], (j, 1, 2)))

Eq(d[1], 2*b[0]*b[1])

Eq(d[2], b[1]**2 - 4)

Eq(d[3], 0)

Eq(d[4], 0)

## The Hamiltonian as a Frobenius Stickelberger determinant

In this section we will show that the conservation of the Hamiltonian is a manifestation of the Frobenius Stickelberger determinant formula for Weierstrass elliptic functions in the 4x4 case.

### Waring-Lagrange like interpolation

The following is inspired by the Waring-Lagrange interpolation identity used in the FWM paper. While the algebraic identity may not be excatly Waring-Lagrange interpolation, it is interpolation of sorts.

Recall that:

In [34]:
Q_poly_uv_no_who = (
    Q_poly_uv
    .replace(gamma[j]-rho(z), u(z, mu[j])*v(z, mu[j]))
    .replace(gamma[k]-rho(z), u(z, mu[k])*v(z, mu[k]))
)

Q_poly_uv_no_who
Q_poly_uv_b

Eq(Q(u(z, mu[1])*v(z, mu[1]), u(z, mu[2])*v(z, mu[2])), a[0] + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2)) + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)))

Eq(Q(u(z, mu[1])*v(z, mu[1]), u(z, mu[2])*v(z, mu[2])), rho(z)**2*b[2] + rho(z)*b[1] + b[0])

In [35]:
rho_gam12_no_quad = Eq(
    rho(z)**2*b[2] + rho(z)*b[1] + b[0],

    (-rho(z) + gamma[2])*(-rho(z) + gamma[1])*b[2]

    + 2*(
        (-rho(z) + gamma[1])*(b[0] + b[1]*gamma[2] + b[2]*gamma[2]**2)/2/(gamma[1] - gamma[2])
        - (-rho(z) + gamma[2])*(b[0] + b[1]*gamma[1] + b[2]*gamma[1]**2)/2/(gamma[1] - gamma[2])
    )
)

assert (rho_gam12_no_quad.lhs - rho_gam12_no_quad.rhs).simplify() == 0, 'rho_gam12_no_quad badly defined'

rho_gam12_no_quad

Eq(rho(z)**2*b[2] + rho(z)*b[1] + b[0], (-rho(z) + gamma[1])*(-rho(z) + gamma[2])*b[2] + (-rho(z) + gamma[1])*(b[0] + b[1]*gamma[2] + b[2]*gamma[2]**2)/(gamma[1] - gamma[2]) - (-rho(z) + gamma[2])*(b[0] + b[1]*gamma[1] + b[2]*gamma[1]**2)/(gamma[1] - gamma[2]))

which in turn can be written in terms of $\rho, \rho'$ as:

In [36]:
rhop_gamma_js = [(rhop_mu_j_gamma_b.subs(j,_j+1).rhs, rhop_mu_j_gamma_b.subs(j,_j+1).lhs) for _j in range(2)]
rho_gamma_js = [(rho_mu_j_gamma.subs(j,_j+1).rhs, rho_mu_j_gamma.subs(j,_j+1).lhs) for _j in range(2)]
Q_in_rho_rhop_only = Eq(
    Q_poly_uv_no_who.rhs, 
    Q_poly_uv_no_who.lhs.subs([
        Q_poly_uv_b.args,
        rho_gam12_no_quad.args
    ])
    .subs(rhop_gamma_js)
    .subs(rho_gamma_js)
)
   
rho_mu_j_gamma
rhop_mu_j_gamma_b
Q_in_rho_rhop_only

Eq(rho(mu[j]), gamma[j])

Eq(\rho'(mu[j]), b[0] + b[1]*gamma[j] + b[2]*gamma[j]**2)

Eq(a[0] + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2)) + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)), (-rho(z) + rho(mu[1]))*(-rho(z) + rho(mu[2]))*b[2] + (-rho(z) + rho(mu[1]))*\rho'(mu[2])/(rho(mu[1]) - rho(mu[2])) - (-rho(z) + rho(mu[2]))*\rho'(mu[1])/(rho(mu[1]) - rho(mu[2])))

In [37]:
rhopz_z1 = drho_wp_d4_b.subs(diff(rho(z),z), rhop(z))
rho_z_min_muj = uv_wp_z0z1muj.subs([uv_j_rho.args, (rho_mu_j_gamma.rhs, rho_mu_j_gamma.lhs)])

rhopz_z1_subs = [
    rhopz_z1.subs(z,mu[_j+1]).args for _j in range(2)
]
rho_z_min_muj_subs = [
    rho_z_min_muj.subs(j,_j+1).args for _j in range(2)
]
rho_muj_min_muk_subs = [
    rho_z_min_muj.subs(z,mu[k]).subs([(j,_j+1), (k,_k+1)]).args 
    for _j in range(2) for _k in range(2) if _j != _k
]
rhopp_muj = [drho_wp_d4_2nd_c.subs(z, mu[1]).args, drho_wp_d4_2nd_c.subs(z, mu[2]).args]
all_rho_rhop_subs = rhopz_z1_subs + rho_z_min_muj_subs + rho_muj_min_muk_subs + rhopp_muj

rho_gam12_rho_wp_wpp = Q_in_rho_rhop_only.subs(all_rho_rhop_subs)
rho_gam12_rho_wp_wpp = Eq(
    rho_gam12_rho_wp_wpp.lhs,
    sum(_.simplify() for _ in rho_gam12_rho_wp_wpp.rhs.args)
)

rho_gam12_rho_wp_wpp

Eq(a[0] + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2)) + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2)), -(wp(z - z0, g2, g3) - wp(-z0 + mu[1], g2, g3))*\wp'(z1, g2, g3)*\wp'(-z0 + mu[2], g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*(wp(-z0 + mu[1], g2, g3) - wp(-z0 + mu[2], g2, g3))*b[2]) + (wp(z - z0, g2, g3) - wp(-z0 + mu[2], g2, g3))*\wp'(z1, g2, g3)*\wp'(-z0 + mu[1], g2, g3)/((wp(z1, g2, g3) - wp(z - z0, g2, g3))*(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*(wp(-z0 + mu[1], g2, g3) - wp(-z0 + mu[2], g2, g3))*b[2]) + (wp(z - z0, g2, g3) - wp(-z0 + mu[1], g2, g3))*(wp(z - z0, g2, g3) - wp(-z0 + mu[2], g2, g3))*\wp'(z1, g2, g3)**2/((wp(z1, g2, g3) - wp(z - z0, g2, g3))**2*(wp(z1, g2, g3) - wp(-z0 + mu[1], g2, g3))*(wp(z1, g2, g3) - wp(-z0 + mu[2], g2, g3))*b[2]))

### The quasi-periodicity of the product of modes and associated identities

Recall that:

In [38]:
u_prod_FS = Eq(0, a_0_eq.lhs - a_0_eq.rhs + duvj.rhs - duvj.lhs)
_u_prod_term = Product(u(z, mu[j]), (j, 1, 2))
u_prod_FS = Eq(
    u_prod_FS.lhs/2 + _u_prod_term,
    u_prod_FS.rhs/2 + _u_prod_term,
)

u_prod_FS

Eq(Product(u(z, mu[j]), (j, 1, 2)), -Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2)

The right hand side is expressible in terms of $\wp, \wp'$ and thus invariant under $z \rightarrow 2m\omega_3+2n\omega_1$. So too, therefore, is the left hand side, which implies:

In [39]:
mu_j_nu_j_z0 = Eq(mu[j], nu[j]+z0)

_sig_log_exps = (
    exp(r[1,1]*log(sigma(z-z0+z1,g2,g3)/sigma(z-z0-z1,g2,g3)))*
    exp(r[1,2]*log(sigma(z-z0+z1,g2,g3)/sigma(z-z0-z1,g2,g3)))
)
sig_log_exps_r1j = Eq(
    _sig_log_exps,
    _sig_log_exps
    .subs(r[1,2], solve(sum_r1j_1.doit(), r[1,2])[0])
    .simplify()
)
u_prod_ham_term = Eq(
    Product(u(z, mu[j]), (j, 1, 2)), 
    Product(u(z, mu[j]).subs(*u_sqrt_wp_z0_z1.args), (j, 1, 2)).replace(r[1,j],0)
    *sig_log_exps_r1j.rhs
)

omega_shift_nm = 2*m*omega[3]+2*n*omega[1]
u_prod_ham_term_shifted = Eq(
    u_prod_ham_term.lhs
    .subs(z,z+omega_shift_nm),
    u_prod_ham_term.rhs
    .subs(z,z+omega_shift_nm)
    .replace(*quasi_period_sigma_b.subs(z,z-2*z0+mu[j]).args)
    .replace(*quasi_period_sigma_b.subs(z,z-z0).args)
    .replace(wp(z-z0+omega_shift_nm, g2,g3), wp(z-z0, g2,g3))
)
u_prod_shited_ratio = Eq(
    u_prod_ham_term.lhs/u_prod_ham_term_shifted.lhs,
    (u_prod_ham_term.rhs/u_prod_ham_term_shifted.rhs)
    .replace(*mu_j_nu_j_z0.args)
    .doit().simplify()
    .subs([
        quasi_period_sigma_b.subs(z, z - z0 - z1).args,
        quasi_period_sigma_b.subs(z, z - z0 + z1).args
    ])
    .simplify()
)

# SomeInteger(n,m)  means the value is some integer but it changes depending on n,m
r_0j_omega = Eq(
    -2*zeta(m*omega[3] + n*omega[1], g2, g3)*(2*z1 + Sum(nu[j], (j, 1, 2)))
    - (2*m*omega[3] + 2*n*omega[1])*Sum(r[0, j], (j, 1, 2)),
    2*I*pi*SomeInteger(n,m)
)

assert (u_prod_shited_ratio.rhs/exp(r_0j_omega.lhs)).simplify().simplify() == 1, 'exp terms not equal'


sig_log_exps_r1j
u_prod_ham_term

mu_j_nu_j_z0
quasi_period_sigma_b
u_prod_shited_ratio
r_0j_omega

Eq(exp(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, 1])*exp(log(sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))*r[1, 2]), sigma(z - z0 + z1, g2, g3)/sigma(z - z0 - z1, g2, g3))

Eq(Product(u(z, mu[j]), (j, 1, 2)), sigma(z - z0 + z1, g2, g3)*Product(sqrt(\wp'(z1, g2, g3))*sigma(z - 2*z0 + mu[j], g2, g3)*exp(z*r[0, j] + epsilon[j])/(sqrt(wp(z1, g2, g3) - wp(z - z0, g2, g3))*sqrt(wp(z1, g2, g3) - wp(-z0 + mu[j], g2, g3))*sigma(z - z0, g2, g3)*sigma(-z0 + mu[j], g2, g3)*sqrt(b[2])), (j, 1, 2))/sigma(z - z0 - z1, g2, g3))

Eq(mu[j], z0 + nu[j])

Eq(sigma(2*m*omega[3] + 2*n*omega[1] + z, g2, g3), (-1)**(m*n + m + n)*sigma(z, g2, g3)*exp((2*m*omega[3] + 2*n*omega[1] + 2*z)*zeta(m*omega[3] + n*omega[1], g2, g3)))

Eq(Product(u(z, mu[j]), (j, 1, 2))/Product(u(2*m*omega[3] + 2*n*omega[1] + z, mu[j]), (j, 1, 2)), exp(-2*m*omega[3]*r[0, 1] - 2*m*omega[3]*r[0, 2] - 2*n*omega[1]*r[0, 1] - 2*n*omega[1]*r[0, 2] - 4*z1*zeta(m*omega[3] + n*omega[1], g2, g3) - 2*zeta(m*omega[3] + n*omega[1], g2, g3)*nu[1] - 2*zeta(m*omega[3] + n*omega[1], g2, g3)*nu[2]))

Eq(-2*(2*z1 + Sum(nu[j], (j, 1, 2)))*zeta(m*omega[3] + n*omega[1], g2, g3) - (2*m*omega[3] + 2*n*omega[1])*Sum(r[0, j], (j, 1, 2)), 2*I*pi*SomeInteger(n, m))

Which, by quasi-periodic properties of $\zeta$ shows that $\nu_1 + \nu_2 + 2 z_1 = 0$ modulo the lattice:

In [40]:
r0j_omega1 = r_0j_omega.subs([(m,0),(n,1)])
r0j_omega3 = r_0j_omega.subs([(m,1),(n,0)])
r0j_nuj = Eq(
    (r0j_omega1.lhs*omega[3] - r0j_omega3.lhs*omega[1])
    .expand().collect(Sum(nu[j],(j,1,4)), factor)
    .subs([zw_1_2_i_pi_omega.args])/(I*pi),
    ((r0j_omega1.rhs*omega[3] - r0j_omega3.rhs*omega[1])/(I*pi)).expand()
)

r0j_omega1
r0j_omega3
zw_1_2_i_pi_omega
r0j_nuj

Eq(-2*(2*z1 + Sum(nu[j], (j, 1, 2)))*zeta(omega[1], g2, g3) - 2*omega[1]*Sum(r[0, j], (j, 1, 2)), 2*I*pi*SomeInteger(1, 0))

Eq(-2*(2*z1 + Sum(nu[j], (j, 1, 2)))*zeta(omega[3], g2, g3) - 2*omega[3]*Sum(r[0, j], (j, 1, 2)), 2*I*pi*SomeInteger(0, 1))

Eq(-zeta(omega[1], g2, g3)*omega[3] + zeta(omega[3], g2, g3)*omega[1], I*pi/2)

Eq(2*z1 + Sum(nu[j], (j, 1, 2)), -2*SomeInteger(0, 1)*omega[1] + 2*SomeInteger(1, 0)*omega[3])

This then shows that the sum over $r_{0,j}$ can be written in terms of $\zeta(\omega_i)$ which allows us to absorb the exponential $z$ term into $\sigma$:

In [41]:
r0j_nuj_b = Eq(
    (
        (r0j_omega1.lhs*omega[3] + r0j_omega3.lhs*omega[1]).subs(Sum(nu[j],(j,1,2)), x - 2*z1)
        .expand().collect(x, factor).subs(x, Sum(nu[j],(j,1,2)) + 2*z1)
        .subs([zw_1_2_i_pi_omega.args])/(I*pi)
    ).subs(*r0j_nuj.args)
    ,
    ((r0j_omega1.rhs*omega[3] + r0j_omega3.rhs*omega[1])/(I*pi)).expand()
)
zw_some_int_id = Eq(2*zw_m_n_omega.rhs, 2*zw_m_n_omega.lhs).subs([
    (m, SomeInteger(1,0)),(n, -SomeInteger(0,1))
])
r0j_nuj_b = Eq(
    Sum(r[0,j],(j,1,2)), 
    solve(r0j_nuj_b, Sum(r[0,j],(j,1,2)))[0].expand()
    .subs(I*pi, solve(zw_1_2_i_pi_omega, I*pi)[0])
    .expand()
    .subs([zw_some_int_id.args])
)

z1_omega = Eq(z1, solve(r0j_nuj.doit(), z1)[0])

sig2_nuj = Eq(sigma(z + z1,g2,g3), sigma(z + z1,g2,g3).subs([z1_omega.args]))
some_int_n = Eq(SomeInteger(0,1), -n)
some_int_m = Eq(SomeInteger(1,0), m)
some_int_subs = [
    some_int_n.args,
    some_int_m.args
]
sig2_nuj_b = sig2_nuj.subs(some_int_subs)
r0j_nuj_c = r0j_nuj_b.subs(some_int_subs)
z1_omega_b = z1_omega.subs(some_int_subs)
quasi_period_sigma_b_z1 = quasi_period_sigma_b.subs(z, z - z1_omega_b.rhs - nu[1] - nu[2])

sig2_nuj_c = sig2_nuj_b.subs([
    quasi_period_sigma_b_z1.args,
    (-r0j_nuj_c.rhs/2, -r0j_nuj_c.lhs/2),
    (m*omega[3] + n*omega[1], solve(z1_omega_b, m*omega[3] + n*omega[1])[0])
])
sig2_nuj_d = Eq(
    sig2_nuj_c.lhs/sig2_nuj_c.lhs.subs(z,0)*exp(z*Sum(r[0,j],(j,1,2))), 
    (sig2_nuj_c.rhs/sig2_nuj_c.rhs.subs(z,0)*exp(z*Sum(r[0,j],(j,1,2))))
    .simplify()
    .subs(sigma(- nu[1] - nu[2] - z1,g2,g3), -sigma(nu[1] + nu[2] + z1,g2,g3))
)

zw_some_int_id
r0j_nuj_b

some_int_n
some_int_m
r0j_nuj_c
z1_omega_b

sig2_nuj_c
sig2_nuj_d

Eq(-2*SomeInteger(0, 1)*zeta(omega[1], g2, g3) + 2*SomeInteger(1, 0)*zeta(omega[3], g2, g3), 2*zeta(-SomeInteger(0, 1)*omega[1] + SomeInteger(1, 0)*omega[3], g2, g3))

Eq(Sum(r[0, j], (j, 1, 2)), -2*zeta(-SomeInteger(0, 1)*omega[1] + SomeInteger(1, 0)*omega[3], g2, g3))

Eq(SomeInteger(0, 1), -n)

Eq(SomeInteger(1, 0), m)

Eq(Sum(r[0, j], (j, 1, 2)), -2*zeta(m*omega[3] + n*omega[1], g2, g3))

Eq(z1, m*omega[3] + n*omega[1] - nu[1]/2 - nu[2]/2)

Eq(sigma(z + z1, g2, g3), (-1)**(m*n + m + n)*sigma(z - z1 - nu[1] - nu[2], g2, g3)*exp(-(2*z - nu[1] - nu[2])*Sum(r[0, j], (j, 1, 2))/2))

Eq(sigma(z + z1, g2, g3)*exp(z*Sum(r[0, j], (j, 1, 2)))/sigma(z1, g2, g3), -sigma(z - z1 - nu[1] - nu[2], g2, g3)/sigma(z1 + nu[1] + nu[2], g2, g3))

### Combining the formulas derived thus far

We combine the formulas derived thus far and insert them into:

In [42]:
u_prod_FS

Eq(Product(u(z, mu[j]), (j, 1, 2)), -Derivative(u(z, mu[j])*v(z, mu[j]), z)/2 + a[0]/2 + Sum(u(z, mu[j])*v(z, mu[j])*a[j], (j, 1, 2))/2 + Sum(u(z, mu[j])*u(z, mu[k])*v(z, mu[j])*v(z, mu[k])*a[j, k]/2, (j, 1, 2), (k, 1, 2))/2)

we evaluate the elliptic function formula that results at the point $z\rightarrow z+z_0$ and multiply both sides by:

In [43]:
wp_z_z1_sqrtd4_term = (wp(z1,g2,g3)-wp(z,g2,g3))**2*b[2]/wpp(z1, g2, g3)
wp_z_z1_sqrtd4_term

(-wp(z, g2, g3) + wp(z1, g2, g3))**2*b[2]/\wp'(z1, g2, g3)

to obtain:

In [44]:
duvj_wpp = Eq(
    duvj.lhs, 
    duvj.lhs.subs([uv_j_rho.args])
    .doit()
    .subs([drho_wp_d4_b.args])
)

u_prod_ham_duv_wp = u_prod_FS.subs([
    u_prod_ham_term.args,
    duvj_wpp.args,
    (rho_gam12_rho_wp_wpp.lhs/2, rho_gam12_rho_wp_wpp.rhs/2),
    (z,z+z0),
    mu_j_nu_j_z0.subs(j,1).args,
    mu_j_nu_j_z0.subs(j,2).args,
    mu_j_nu_j_z0.subs(j,3).args,
]).replace(*mu_j_nu_j_z0.args)


u_prod_ham_duv_wp = Eq(
    (
        u_prod_ham_duv_wp.lhs.doit()*wp_z_z1_sqrtd4_term*sig2_nuj_d.rhs/sig2_nuj_d.lhs
    ).expand().simplify().simplify().subs([sigma_p_identity.subs([(y,z1),(x,z)]).args]),
    sum(_*wp_z_z1_sqrtd4_term for _ in u_prod_ham_duv_wp.rhs.args)
)

u_prod_ham_duv_wp

Eq(-sigma(z + z1, g2, g3)*sigma(z + nu[1], g2, g3)*sigma(z + nu[2], g2, g3)*sigma(z - z1 - nu[1] - nu[2], g2, g3)*exp(z0*r[0, 1] + z0*r[0, 2] + epsilon[1] + epsilon[2])/(sqrt(wp(z1, g2, g3) - wp(nu[1], g2, g3))*sqrt(wp(z1, g2, g3) - wp(nu[2], g2, g3))*sigma(z, g2, g3)**4*sigma(z1, g2, g3)*sigma(z1 + nu[1] + nu[2], g2, g3)*sigma(nu[1], g2, g3)*sigma(nu[2], g2, g3)), -(-wp(z, g2, g3) + wp(z1, g2, g3))*(wp(z, g2, g3) - wp(nu[1], g2, g3))*\wp'(nu[2], g2, g3)/(2*(wp(z1, g2, g3) - wp(nu[2], g2, g3))*(wp(nu[1], g2, g3) - wp(nu[2], g2, g3))) + (-wp(z, g2, g3) + wp(z1, g2, g3))*(wp(z, g2, g3) - wp(nu[2], g2, g3))*\wp'(nu[1], g2, g3)/(2*(wp(z1, g2, g3) - wp(nu[1], g2, g3))*(wp(nu[1], g2, g3) - wp(nu[2], g2, g3))) + (wp(z, g2, g3) - wp(nu[1], g2, g3))*(wp(z, g2, g3) - wp(nu[2], g2, g3))*\wp'(z1, g2, g3)/(2*(wp(z1, g2, g3) - wp(nu[1], g2, g3))*(wp(z1, g2, g3) - wp(nu[2], g2, g3))) + \wp'(z, g2, g3)/2)

We multiply both sides by $\sigma(z)^4$ and let $z \rightarrow 0$ to obtain:

In [45]:
pw_sigma_zs_ids = [
    sigma_p_identity.subs([(y,z),(x,nu[j])]),
    sigma_p_identity.subs([(y,z),(x,nu[k])]),
    sigma_p_identity.subs([(y,z1),(x,nu[j])]),
    sigma_p_identity.subs([(y,z1),(x,nu[k])]),
    sigma_p_identity.subs([(y,z1),(x,z)])
]

all_wp_sigma_j_subs = [pw_sigma_zs_ids[0].subs(j,_j+1).args for _j in range(2)] + [pwp_sigma_dbl_ratio.args]

u_prod_ham_duv_wp_z_0 = Eq(
    (u_prod_ham_duv_wp.lhs*sigma(z,g2,g3)**4)
    .subs(z,0).simplify()
    .subs(sigma(-z1-nu[1]-nu[2], g2, g3), -sigma(z1+nu[1]+nu[2], g2, g3)),
    sum(
        (_*sigma(z,g2,g3)**4).simplify() 
        for _ in u_prod_ham_duv_wp.rhs.subs(all_wp_sigma_j_subs).args
    )
).subs([(z,0),(sigma(0,g2,g3),0)])


u_prod_ham_duv_wp_z_0

Eq(exp(z0*r[0, 1] + z0*r[0, 2] + epsilon[1] + epsilon[2])/(sqrt(wp(z1, g2, g3) - wp(nu[1], g2, g3))*sqrt(wp(z1, g2, g3) - wp(nu[2], g2, g3))), \wp'(z1, g2, g3)/(2*(wp(z1, g2, g3) - wp(nu[1], g2, g3))*(wp(z1, g2, g3) - wp(nu[2], g2, g3))))

Substituting the term at $z=0$ into the previous equation and multiplying through to clear denominators on the right hand side gives:

In [46]:
wp_nu123_denom = fraction(u_prod_ham_duv_wp_z_0.rhs.simplify())[1]
u_prod_ham_duv_wp_b = Eq(
    (u_prod_ham_duv_wp.lhs/u_prod_ham_duv_wp_z_0.lhs)
    .simplify()*
    (u_prod_ham_duv_wp_z_0.rhs)
    .simplify()
    *wp_nu123_denom,
    (u_prod_ham_duv_wp.rhs*wp_nu123_denom)
    .expand()
    .collect([wpp(nu[1], g2,g3), wpp(nu[2], g2,g3), wpp(z1, g2,g3), wpp(z, g2,g3)], factor)
)
_left_over_denom = fraction(u_prod_ham_duv_wp_b.rhs.simplify())[1]
u_prod_ham_duv_wp_b = Eq(
    u_prod_ham_duv_wp_b.lhs*_left_over_denom,
    (u_prod_ham_duv_wp_b.rhs*_left_over_denom)
    .expand()
    .collect([wpp(nu[1], g2,g3), wpp(nu[2], g2,g3), wpp(z1, g2,g3), wpp(z, g2,g3)], factor)
)

u_prod_ham_duv_wp_b

Eq(-(wp(nu[1], g2, g3) - wp(nu[2], g2, g3))*\wp'(z1, g2, g3)*sigma(z + z1, g2, g3)*sigma(z + nu[1], g2, g3)*sigma(z + nu[2], g2, g3)*sigma(z - z1 - nu[1] - nu[2], g2, g3)/(sigma(z, g2, g3)**4*sigma(z1, g2, g3)*sigma(z1 + nu[1] + nu[2], g2, g3)*sigma(nu[1], g2, g3)*sigma(nu[2], g2, g3)), (-wp(z, g2, g3) + wp(nu[1], g2, g3))*(-wp(z, g2, g3) + wp(nu[2], g2, g3))*(wp(nu[1], g2, g3) - wp(nu[2], g2, g3))*\wp'(z1, g2, g3) + (-wp(z, g2, g3) + wp(nu[1], g2, g3))*(wp(z, g2, g3) - wp(z1, g2, g3))*(-wp(z1, g2, g3) + wp(nu[1], g2, g3))*\wp'(nu[2], g2, g3) - (-wp(z, g2, g3) + wp(nu[2], g2, g3))*(wp(z, g2, g3) - wp(z1, g2, g3))*(-wp(z1, g2, g3) + wp(nu[2], g2, g3))*\wp'(nu[1], g2, g3) + (-wp(z1, g2, g3) + wp(nu[1], g2, g3))*(-wp(z1, g2, g3) + wp(nu[2], g2, g3))*(wp(nu[1], g2, g3) - wp(nu[2], g2, g3))*\wp'(z, g2, g3))

### The Frobenius Stickelberger determinant formula

The FS formula tells us that, in general:

In [47]:
def p_matrix_f_s(N): 
    return Matrix([[1, *[pw(x,g2,g3).diff((x,j)).subs(x,mu[k]) for j in range(N)]] for k in range(N+1)])

sigma_product_f_s = (
    (-1)**(N*(N-1)/2)*
    Product(factorial(k),(k,1,N))*
    sigma(Sum(mu[j],(j,0,N)),g2,g3)*
    Product(Product(Piecewise((sigma(mu[l] - mu[m], g2, g3),l<m),(1,True)),(l,0,N)),(m,0,N))/
    Product(sigma(mu[n],g2,g3)**(N+1),(n,0,N))
)

def frob_stick(Nval, evaluated=False, WPversion=False):
    if evaluated:
        return Eq(p_matrix_f_s(Nval).det(), sigma_product_f_s.subs(N,Nval).doit())
    elif WPversion:
        return Eq(WPdet(Nval), sigma_product_f_s.subs(N,Nval).doit())
    return Eq(Det(p_matrix_f_s(Nval)), sigma_product_f_s.subs(N,Nval).doit())

fs_original_4_uneval = frob_stick(3, evaluated=False, WPversion=False)


ho_diff_subs = [
    _eq.subs(z, mu[_j]).args for _eq in [wpk(_i) for _i in range(1,6)] for _j in range(5)
][::-1]

# for _args in ho_diff_subs:
#     Eq(*_args)

fs_original_4_eval = (
    frob_stick(3, evaluated=True, WPversion=False)
    .subs(ho_diff_subs)
    .replace(pw,wp).replace(pwp,wpp)
)
fs_original_4_eval = Eq(
    fs_original_4_eval.lhs
    .expand().collect([wpp(mu[_j], g2, g3) for _j in range(7)], factor),
    fs_original_4_eval.rhs
)

fs_original_4_uneval
fs_original_4_eval

Eq(Det(Matrix([
[1, pw(mu[0], g2, g3), Derivative(pw(mu[0], g2, g3), mu[0]), Derivative(pw(mu[0], g2, g3), (mu[0], 2))],
[1, pw(mu[1], g2, g3), Derivative(pw(mu[1], g2, g3), mu[1]), Derivative(pw(mu[1], g2, g3), (mu[1], 2))],
[1, pw(mu[2], g2, g3), Derivative(pw(mu[2], g2, g3), mu[2]), Derivative(pw(mu[2], g2, g3), (mu[2], 2))],
[1, pw(mu[3], g2, g3), Derivative(pw(mu[3], g2, g3), mu[3]), Derivative(pw(mu[3], g2, g3), (mu[3], 2))]])), -12*sigma(mu[0] - mu[1], g2, g3)*sigma(mu[0] - mu[2], g2, g3)*sigma(mu[0] - mu[3], g2, g3)*sigma(mu[1] - mu[2], g2, g3)*sigma(mu[1] - mu[3], g2, g3)*sigma(mu[2] - mu[3], g2, g3)*sigma(mu[0] + mu[1] + mu[2] + mu[3], g2, g3)/(sigma(mu[0], g2, g3)**4*sigma(mu[1], g2, g3)**4*sigma(mu[2], g2, g3)**4*sigma(mu[3], g2, g3)**4))

Eq(6*(wp(mu[0], g2, g3) - wp(mu[1], g2, g3))*(wp(mu[0], g2, g3) - wp(mu[2], g2, g3))*(wp(mu[1], g2, g3) - wp(mu[2], g2, g3))*\wp'(mu[3], g2, g3) - 6*(wp(mu[0], g2, g3) - wp(mu[1], g2, g3))*(wp(mu[0], g2, g3) - wp(mu[3], g2, g3))*(wp(mu[1], g2, g3) - wp(mu[3], g2, g3))*\wp'(mu[2], g2, g3) + 6*(wp(mu[0], g2, g3) - wp(mu[2], g2, g3))*(wp(mu[0], g2, g3) - wp(mu[3], g2, g3))*(wp(mu[2], g2, g3) - wp(mu[3], g2, g3))*\wp'(mu[1], g2, g3) - 6*(wp(mu[1], g2, g3) - wp(mu[2], g2, g3))*(wp(mu[1], g2, g3) - wp(mu[3], g2, g3))*(wp(mu[2], g2, g3) - wp(mu[3], g2, g3))*\wp'(mu[0], g2, g3), -12*sigma(mu[0] - mu[1], g2, g3)*sigma(mu[0] - mu[2], g2, g3)*sigma(mu[0] - mu[3], g2, g3)*sigma(mu[1] - mu[2], g2, g3)*sigma(mu[1] - mu[3], g2, g3)*sigma(mu[2] - mu[3], g2, g3)*sigma(mu[0] + mu[1] + mu[2] + mu[3], g2, g3)/(sigma(mu[0], g2, g3)**4*sigma(mu[1], g2, g3)**4*sigma(mu[2], g2, g3)**4*sigma(mu[3], g2, g3)**4))

in the $z, \nu_j$ notation this is:

In [48]:
mu0j_to_nu1js = [
    (mu[0], z),
    (mu[1], -nu[1]),
    (mu[2], -nu[2]),
    (mu[3], -nu[3])
]
wp_signs_mujs = [(wp(-nu[_j+1], g2, g3), wp(nu[_j+1], g2, g3)) for _j in range(4)]
wpp_signs_mujs = [(wpp(-nu[_j+1], g2, g3), -wpp(nu[_j+1], g2, g3)) for _j in range(4)]
sigma_signs_mujs = [(sigma(-nu[_j+1], g2, g3), -sigma(nu[_j+1], g2, g3)) for _j in range(4)]
sigma_sign_diffs_mujs = [
    (sigma(-nu[1] + nu[2], g2, g3), -sigma(nu[1] - nu[2], g2, g3)),
    (sigma(-nu[1] + nu[3], g2, g3), -sigma(nu[1] - nu[3], g2, g3)),
    (sigma(-nu[2] + nu[3], g2, g3), -sigma(nu[2] - nu[3], g2, g3))
]

fs_original_4_nu = Eq(
    fs_original_4_eval.rhs, 
    fs_original_4_eval.lhs
).subs(mu0j_to_nu1js).subs(
    wp_signs_mujs +
    wpp_signs_mujs +
    sigma_signs_mujs +
    sigma_sign_diffs_mujs
).subs(nu[3], z1)

fs_original_4_nu

Eq(12*sigma(z + z1, g2, g3)*sigma(z + nu[1], g2, g3)*sigma(z + nu[2], g2, g3)*sigma(-z1 + nu[1], g2, g3)*sigma(-z1 + nu[2], g2, g3)*sigma(nu[1] - nu[2], g2, g3)*sigma(z - z1 - nu[1] - nu[2], g2, g3)/(sigma(z, g2, g3)**4*sigma(z1, g2, g3)**4*sigma(nu[1], g2, g3)**4*sigma(nu[2], g2, g3)**4), 6*(wp(z, g2, g3) - wp(z1, g2, g3))*(wp(z, g2, g3) - wp(nu[1], g2, g3))*(-wp(z1, g2, g3) + wp(nu[1], g2, g3))*\wp'(nu[2], g2, g3) - 6*(wp(z, g2, g3) - wp(z1, g2, g3))*(wp(z, g2, g3) - wp(nu[2], g2, g3))*(-wp(z1, g2, g3) + wp(nu[2], g2, g3))*\wp'(nu[1], g2, g3) - 6*(wp(z, g2, g3) - wp(nu[1], g2, g3))*(wp(z, g2, g3) - wp(nu[2], g2, g3))*(wp(nu[1], g2, g3) - wp(nu[2], g2, g3))*\wp'(z1, g2, g3) - 6*(-wp(z1, g2, g3) + wp(nu[1], g2, g3))*(-wp(z1, g2, g3) + wp(nu[2], g2, g3))*(wp(nu[1], g2, g3) - wp(nu[2], g2, g3))*\wp'(z, g2, g3))

from which, by again mulitplying by $\sigma(z)^4$ and letting $z \rightarrow 0$, we obtain:

In [49]:
fs4_b = fs_original_4_nu.subs(all_wp_sigma_j_subs)
fs4_c = Eq(
    fs4_b.lhs*sigma(z, g2, g3)**4,
    sum(_*sigma(z, g2, g3)**4 for _ in fs4_b.rhs.args),
).subs([(z,0), (sigma(0,g2,g3),0), (sigma(-nu[1]-nu[2]-nu[3],g2,g3), -sigma(nu[1]+nu[2]+nu[3],g2,g3))])

fs4_c

Eq(12*sigma(-z1 + nu[1], g2, g3)*sigma(-z1 + nu[2], g2, g3)*sigma(nu[1] - nu[2], g2, g3)*sigma(-z1 - nu[1] - nu[2], g2, g3)/(sigma(z1, g2, g3)**3*sigma(nu[1], g2, g3)**3*sigma(nu[2], g2, g3)**3), -6*(wp(nu[1], g2, g3) - wp(nu[2], g2, g3))*\wp'(z1, g2, g3))

substituting that identity into the final formula of the previous section confirms that the underlying identity behind the elliptic function dynamics is the Frobenius Stickelberger determinant:

In [50]:
_eq1 = Eq(
    fs_original_4_nu.lhs,
    sum(_.factor() for _ in fs_original_4_nu.rhs.args)
)

_eq2 = Eq(
    -6*(u_prod_ham_duv_wp_b.lhs*fs4_c.lhs/fs4_c.rhs)
    .subs([
        (sigma(- z1 - nu[1] - nu[2], g2, g3), -sigma(z1 + nu[1] + nu[2], g2, g3))
    ]),
    -sum(6*_.factor() for _ in u_prod_ham_duv_wp_b.rhs.args)
)

assert (_eq1.lhs - _eq2.lhs).simplify() == 0, 'fs_lhs_check failed'
assert (_eq1.rhs - _eq2.rhs).simplify() == 0, 'fs_lhs_check failed'

_eq1
_eq2

Eq(12*sigma(z + z1, g2, g3)*sigma(z + nu[1], g2, g3)*sigma(z + nu[2], g2, g3)*sigma(-z1 + nu[1], g2, g3)*sigma(-z1 + nu[2], g2, g3)*sigma(nu[1] - nu[2], g2, g3)*sigma(z - z1 - nu[1] - nu[2], g2, g3)/(sigma(z, g2, g3)**4*sigma(z1, g2, g3)**4*sigma(nu[1], g2, g3)**4*sigma(nu[2], g2, g3)**4), -6*(-wp(z, g2, g3) + wp(nu[1], g2, g3))*(-wp(z, g2, g3) + wp(nu[2], g2, g3))*(wp(nu[1], g2, g3) - wp(nu[2], g2, g3))*\wp'(z1, g2, g3) - 6*(-wp(z, g2, g3) + wp(nu[1], g2, g3))*(wp(z, g2, g3) - wp(z1, g2, g3))*(-wp(z1, g2, g3) + wp(nu[1], g2, g3))*\wp'(nu[2], g2, g3) + 6*(-wp(z, g2, g3) + wp(nu[2], g2, g3))*(wp(z, g2, g3) - wp(z1, g2, g3))*(-wp(z1, g2, g3) + wp(nu[2], g2, g3))*\wp'(nu[1], g2, g3) - 6*(-wp(z1, g2, g3) + wp(nu[1], g2, g3))*(-wp(z1, g2, g3) + wp(nu[2], g2, g3))*(wp(nu[1], g2, g3) - wp(nu[2], g2, g3))*\wp'(z, g2, g3))

Eq(12*sigma(z + z1, g2, g3)*sigma(z + nu[1], g2, g3)*sigma(z + nu[2], g2, g3)*sigma(-z1 + nu[1], g2, g3)*sigma(-z1 + nu[2], g2, g3)*sigma(nu[1] - nu[2], g2, g3)*sigma(z - z1 - nu[1] - nu[2], g2, g3)/(sigma(z, g2, g3)**4*sigma(z1, g2, g3)**4*sigma(nu[1], g2, g3)**4*sigma(nu[2], g2, g3)**4), -6*(-wp(z, g2, g3) + wp(nu[1], g2, g3))*(-wp(z, g2, g3) + wp(nu[2], g2, g3))*(wp(nu[1], g2, g3) - wp(nu[2], g2, g3))*\wp'(z1, g2, g3) - 6*(-wp(z, g2, g3) + wp(nu[1], g2, g3))*(wp(z, g2, g3) - wp(z1, g2, g3))*(-wp(z1, g2, g3) + wp(nu[1], g2, g3))*\wp'(nu[2], g2, g3) + 6*(-wp(z, g2, g3) + wp(nu[2], g2, g3))*(wp(z, g2, g3) - wp(z1, g2, g3))*(-wp(z1, g2, g3) + wp(nu[2], g2, g3))*\wp'(nu[1], g2, g3) - 6*(-wp(z1, g2, g3) + wp(nu[1], g2, g3))*(-wp(z1, g2, g3) + wp(nu[2], g2, g3))*(wp(nu[1], g2, g3) - wp(nu[2], g2, g3))*\wp'(z, g2, g3))

## Integral of $\rho(z)$ (useful for gauge transforms)

In [51]:
rho_zeta_muj = Eq(
    rho_zeta.lhs.subs(z, mu[j]) - rho_zeta.lhs,
    (rho_zeta.rhs.subs(z, mu[j]) - rho_zeta.rhs)
    .simplify()
    .subs(zeta(-z+z0+z1,g2,g3), -zeta(z-z0-z1,g2,g3))
)
rho_zeta_muj_logs = rho_zeta_muj.subs([
    zw_dlog_sigma_identity.subs(x, z0 - z1).args,
    zw_dlog_sigma_identity.subs(x, z0 + z1).args
])
rho_integral = Eq(
    Integral(-rho(z) + rho(mu[j]),z),
    z*(zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/b[2] +
    log(sigma(z - z0 - z1, g2, g3)/sigma(z - z0 + z1, g2, g3))/b[2] + C
)
assert (diff(rho_integral.rhs, z) - rho_zeta_muj_logs.rhs).doit().simplify() == 0, 'rho integral badly defined'
assert (diff(rho_integral.lhs, z) - rho_zeta_muj_logs.lhs).doit().simplify() == 0, 'rho integral badly defined'

rho_zeta
rho_zeta_muj
rho_zeta_muj_logs
rho_integral

Eq(rho(z), -(2*zeta(z1, g2, g3) - zeta(-z + z0 + z1, g2, g3) - zeta(z - z0 + z1, g2, g3))/b[2] + lamda[1])

Eq(-rho(z) + rho(mu[j]), (zeta(z - z0 - z1, g2, g3) - zeta(z - z0 + z1, g2, g3) + zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/b[2])

Eq(-rho(z) + rho(mu[j]), (zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3) + Derivative(log(sigma(z - z0 - z1, g2, g3)), z) - Derivative(log(sigma(z - z0 + z1, g2, g3)), z))/b[2])

Eq(Integral(-rho(z) + rho(mu[j]), z), C + z*(zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3))/b[2] + log(sigma(z - z0 - z1, g2, g3)/sigma(z - z0 + z1, g2, g3))/b[2])

In [52]:
__xtmp = Eq(x, zeta(-z0+z1+mu[j],g2,g3)+zeta(z0+z1-mu[j],g2,g3))
zeta_z0_z1_muj_gam_lam = Eq(
    __xtmp.rhs, 
    solve(rho_zeta.subs([(z, mu[j]), (__xtmp.rhs, __xtmp.lhs)]), __xtmp.lhs)[0]
    .subs([rho_mu_j_gamma.args])
)

rho_integral_b = Eq(
    rho_integral.lhs,
    rho_integral.rhs
    .subs([zeta_z0_z1_muj_gam_lam.args])
    .expand().collect(z)
)


zeta_z0_z1_muj_gam_lam
rho_integral_b

Eq(zeta(-z0 + z1 + mu[j], g2, g3) + zeta(z0 + z1 - mu[j], g2, g3), 2*zeta(z1, g2, g3) + b[2]*gamma[j] - b[2]*lamda[1])

Eq(Integral(-rho(z) + rho(mu[j]), z), C + z*(2*zeta(z1, g2, g3)/b[2] + gamma[j] - lamda[1]) + log(sigma(z - z0 - z1, g2, g3)/sigma(z - z0 + z1, g2, g3))/b[2])